# 02 - Calibration of the Player Generation Model

**Objective**: This notebook aims to extract some statistical laws underlying the carreers of tennis players from ATP results (1991-2024). We will extract parameters from our model to configure the players in our simulation later. 

**Method**: Each player will be attributed a score _S_ determined by an _intrinsic potential P_, calibrated on the distribution of maximum strengths of the players (based on the dataset available), and _aging A_, that is a function describing the evolution of strength with age.

## 0. Creation of Players Database Info

We create a dataset containing the main information about players' careers. We extract the maximum strength achieved by each player (from the `zermelo_strengths_1991-2024.csv` file) and the age at which this maximum strength was reached. We also store the age at first and last match played (We use the year of birth from the `atp_players.csv` file to compute ages). 

Since age is a crucial factor in our model, any player with missing birthdate information is excluded from the dataset. (The impact is minimal, as these kind of players have not played many matches in the dataset and have small Zermelo strengths).

For the players active in the boundaries of the dataset (1991 and 2024), we retain them in the dataset because they provide valuable information about the distribution of strengths. However, they will be excluded from the calibration of aging curves and carreer duration later, since we do not have their full career data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

# Note: scipy.stats is used throughout the code.

In [ ]:
zermelo_strengths_data = pd.read_csv("../data/processed/zermelo_strengths_1991-2024.csv")
players_info_data = pd.read_csv("../data/tennis_atp/atp_players.csv", low_memory=False)
#display(players_info_data.head())

In [ ]:
# sort the data so we can get the year corresponding to the maximum strength on the first line
zermelo_strengths_data_sorted = zermelo_strengths_data.sort_values("zermelo_strength", ascending=False)

# group the data by IDs, and then get: start and end years, max strength and corresponding year
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html
players_stats_data = zermelo_strengths_data_sorted.groupby("player_id").agg(start_year=("year", "min"), 
                                                       end_year=("year", "max"), 
                                                       top_year=("year", "first"),
                                                       top_strength=("zermelo_strength", "max"),
                                                       active_years=("year", "nunique")).reset_index()                     

display(players_stats_data)

In [ ]:
# formatting year of birth to get only the year (with .dt.year)
# https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
# https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior
players_info_data["birth_year"] = pd.to_datetime(players_info_data["dob"].astype(str), format="%Y%m%d.0", errors="coerce").dt.year

In [ ]:
players_full_data = pd.merge(players_stats_data, players_info_data[["player_id","birth_year", "name_first", "name_last"]], on="player_id", how="left")
players_full_data = players_full_data.dropna(subset = ["birth_year"]) # delete all the players without a valid birth year 
players_full_data["birth_year"] = players_full_data["birth_year"].astype(int) # to remove the .0 after the year

# add start_age / top_age / end_age
players_full_data["start_age"]= players_full_data["start_year"]-players_full_data["birth_year"]
players_full_data["end_age"]= players_full_data["end_year"]-players_full_data["birth_year"]
players_full_data["top_age"]= players_full_data["top_year"]-players_full_data["birth_year"]

# removal of players who have a negative age or are younger than 14 years old
# It seems that different players have the same ID, so they are deleted (e.g. son confused with his father)
# example: Martin Damm (father born in 1972, son born in 2003 --> year displayed: 2003)
players_full_data = players_full_data[players_full_data["start_age"]>=14]

display(players_full_data.head())

In [ ]:
#check whether a player was active both in 1991 and in 2024
all_years_player = players_full_data[(players_full_data["start_year"]==1991) & (players_full_data["end_year"]==2024)]
print("Number of players active both in 1991 and in 2024:", len(all_years_player))

# Detecting complete careers: new column added in the data to find the players whose start and end are available 
# (i.e. who started after 1991 and ended before 2024)

players_full_data["career_type"] = np.where((players_full_data["start_year"]>1991) & 
                                                (players_full_data["end_year"]<2024), 
                                                "full", "Other")

players_full_data["career_type"] = np.where(players_full_data["start_year"]==1991, "left_boundary",
                                               players_full_data["career_type"])

players_full_data["career_type"] = np.where(players_full_data["end_year"]==2024, "right_boundary",
                                               players_full_data["career_type"])


full_career_players_nbr = (players_full_data["career_type"]=="full").sum() 
valid_careers_nbr = len(players_full_data)

print("Number of complete careers:", full_career_players_nbr, "out of", valid_careers_nbr, 
      f"({100*full_career_players_nbr/valid_careers_nbr:.2f}%)")

In [ ]:
# rearranging the order of the columns
players_full_data = players_full_data[["player_id", "name_first", "name_last", 
                                       "birth_year", "start_year", "end_year", "top_year", 
                                       "start_age", "end_age", "top_age", 
                                       "top_strength", "career_type", "active_years"]]

In [ ]:
# saving the file in a .csv
output_path = "../data/processed/players_stats.csv"
players_full_data.to_csv(output_path, index=False)

All the information is stored in a new dataframe `players_full_data`. It contains the following columns: $\\$
$\textbf{player\_id, name\_first, name\_last, birth\_year, start\_year, end\_year, top\_year, start\_age}$
$\textbf{end\_age, top\_age, top\_strength, complete\_career}$.

## 1. Testing the Stationarity of the Player Generation Process

Before calibrating our model, we must verify that the process generating players is stationary over time. This verification stands on the hypothesis that the distribution of players' intrinsic potentials (maximum strengths) remains consistent across different time periods, and that the number of new players entering the professional circuit each year is relatively stable. 

### 1.1. Stationarity of Maximum Strengths Distribution (Quality)

We first check that the distribution of players' strengths is stationary over time. We plot the distribution of players' maximum strengths for different time periods and compare them. To validate this hypothesis, we examine the evolution of `top_strength` based on the player's `start_year` by focusing on the average level stability (median) and the elite level stability (top 10% players). The overall shape of distribution over decades is also analysed to ensure no significant shifts occur.

If this hypothesis holds, we can pool all players together to estimate the distribution of intrinsic potentials.

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the median trend line
sns.lineplot(data=players_full_data,
             x="start_year", y="top_strength", 
             estimator="median", errorbar=("pi",50), # colored band containing 50% of the points around the median
             color="#e74c3c",linewidth=3,
             label="Median Strength",
             zorder=3)

# plot of the top 10% trend line
sns.lineplot(data=players_full_data, 
             x="start_year", y="top_strength", 
             estimator=lambda x: np.percentile(x, 90), # 90th percentile = top 10% (value )
             errorbar=("ci", 95), # confidence interval of 95%
             label="90th Percentile Strength",
             color="#8e44ad", linewidth=3, linestyle="--", zorder=4)

# plot of the maximum trend line
sns.lineplot(data=players_full_data, x="start_year", y="top_strength", 
             estimator=np.max, errorbar=None, 
             label="Max Strength",
             color="#f1c40f", linewidth=3, linestyle="-.", zorder=5)


# plot of the individual points
sns.stripplot(data=players_full_data, 
              x="start_year", y="top_strength", 
              size=2.5, 
              hue="career_type",
              hue_order=["full", "left_boundary", "right_boundary"],
              palette=["#687475", "#60a215e8", "#1484cf"], 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)

plt.title("Distribution of Player Maximum Zermelo Strengths by Career Start Year", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12)

plt.yscale("log")
plt.ylabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)

# legend (keep only the relevant legend items)
handles, _ = plt.gca().get_legend_handles_labels()

handles_curves = [handles[0], handles[1], handles[2]]
labels_curves = ["Median (IQR band)", "Top 10% (95% CI)", "Top 1"]
legend_curves = plt.legend(handles=handles_curves, 
                           labels=labels_curves, 
                           markerscale=2.5, fontsize=13,
                           loc="upper right",
                           frameon=False)

plt.gca().add_artist(legend_curves) # to keep both legends


handles_points = [handles[4], handles[3], handles[5]]
labels_points = ["Left bounded (start in 1991)", "Complete Career", "Right bounded (end in 2024)"]
plt.legend(handles=handles_points, 
                           labels=labels_points, 
                           markerscale=3, fontsize=13,
                           bbox_to_anchor=(0.8, -0.15),
                           frameon=True, ncol=3)


plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

<font color="blue"> **_The figure strongly supports our stationary hypothesis!_** </font> The median player strength (in red) and the elite threshold (in purple) remain horizontal and stable from 1992 to 2018, proving that the generation of talent has not changed structurally over time. 

The high variance in 1991 comes from players already in their prime (they started before 1991, but the first year of our dataset is 1991). We observe a slight increase in 2019, immediately followed by a decrease in 2020 (likely due to consequences of the COVID-19 pandemic). After this year, it continues decreasing due to right-censoring: players starting their carreers recently have not yet reached their real maximum strength. 

Regarding the elites, the top 10% threshold mirrors the stabilitiy of the median, except with some small fluctuations. It confirms that the top players can be generated consistently year over year. 

The Top 1 curve (in yellow) is more volatile, with peaks rather than a specific trend. This is expected, as the very best players can vary significantly from year to year due to the emergence of exceptional talents or the dominance of a few players.

---

To ensure robutstness, we must restrict the calibration of our model parameters to a stable period **1992-20YY**, where the stationary hypothesis is most valid. To determine the cut-off year, we estimate the time required for players to reach their potential, by taking players that started between 1992 and 2002.  It will be done by calcutating the years to reach their maximum strength. 

In [ ]:
safe_year_start = 1992
safe_year_end = 2002

In [ ]:
# finding the players that begin between 1992 and 2002
years_to_top_players = players_full_data[(players_full_data["start_year"]>=safe_year_start) & (players_full_data["start_year"]<=safe_year_end)].copy()

# calculating the years to reach top
years_to_top_players["years_to_top"] = years_to_top_players["top_year"]-years_to_top_players["start_year"]

years_to_top_players["years_to_top"].describe(percentiles=[0.25,0.5,0.9, 0.95, 0.99])

The median tie to reach peak strength is only 1 year, and the first quartile is 0 year. This indicates that the majority of players do not experience a long development: they enter the tour, reach their maximum almost immediately and decline. As a result, their top year is simply the year they retired.

To get a realistic estimate of the time needed for a player to reach teir peak, we temporarily filter the data by keeping only players who where **active for at least 5 years**.

In [ ]:
min_active_years = 5

In [ ]:
filtered_years_to_top_players = years_to_top_players[years_to_top_players["active_years"]>= min_active_years].copy()
print(f"Number of players with at least {min_active_years} active years ({safe_year_start}-{safe_year_end}):", len(filtered_years_to_top_players))

In [ ]:
# calculating the years to reach top
filtered_years_to_top_players["years_to_top"] = filtered_years_to_top_players["top_year"]-filtered_years_to_top_players["start_year"]

filtered_years_to_top_players["years_to_top"].describe(percentiles=[0.25,0.5,0.9, 0.95, 0.99])

The maximum value of 22 years seems to be unrealistic. The 90th percentile lies at 10 years. This means that only 10% of players take more than 10 years to reach their peak, which is a reasonable threshold to consider for our cut-off year.

<font color="purple"> Calibration is made on the starting year period **1992-2014**! </font> 
$\newline$ _to ensure that all players (the vast majority) have reached their potential._

In [ ]:
calibration_start = 1992
calibration_end = 2014

Now, we want to see if the distribution of maximum strengths is consistent across generations. We plot the distribution of `top_strength` for players starting in different decades (1990s, 2000s, 2010s).

**Important**: While the 5-year filter was necessary to estimate the development period and establish the cut-off year, we will not apply this filter for the calibration of the intrinsic potentials. To avoid bias, the talent distribution has to reflect all players who entered the circuit.

In [ ]:
# considering only the players in the calibration period
calibration_players = players_full_data[(players_full_data["start_year"]>=calibration_start) & 
                                        (players_full_data["start_year"]<=calibration_end)].copy()

print("Total number of players in the calibration period:", len(calibration_players))

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the median trend line
sns.lineplot(data=calibration_players,
             x="start_year", y="top_strength", 
             estimator="median", errorbar=("pi",50), # colored band containing 50% of the points around the median
             color="#e74c3c",linewidth=3,
             label="Median Strength",
             zorder=3)

# plot of the top 10% trend line
sns.lineplot(data=calibration_players, 
             x="start_year", y="top_strength", 
             estimator=lambda x: np.percentile(x, 90), # 90th percentile = top 10% (value )
             errorbar=("ci", 95), # confidence interval of 95%
             label="90th Percentile Strength",
             color="#8e44ad", linewidth=3, linestyle="--", zorder=4)

# plot of the maximum trend line
sns.lineplot(data=calibration_players, x="start_year", y="top_strength", 
             estimator=np.max, errorbar=None, 
             label="Max Strength",
             color="#f1c40f", linewidth=3, linestyle="-.", zorder=5)


# plot of the individual points
sns.stripplot(data=calibration_players, 
              x="start_year", y="top_strength", 
              size=2.5, 
              hue="career_type",
              hue_order=["full", "left_boundary", "right_boundary"],
              palette=["#687475", "#60a215e8", "#1484cf"], 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)

plt.title("Distribution of Player Maximum Zermelo Strengths by Career Start Year \n(Calibration Period)", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12)

plt.yscale("log")
plt.ylabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)

# legend (keep only the relevant legend items)
handles, _ = plt.gca().get_legend_handles_labels()

handles_curves = [handles[0], handles[1], handles[2]]
labels_curves = ["Median (IQR band)", "Top 10% (95% CI)", "Top 1"]
legend_curves = plt.legend(handles=handles_curves, 
                           labels=labels_curves, 
                           markerscale=2.5, fontsize=13,
                           loc="upper right",
                           frameon=False)

plt.gca().add_artist(legend_curves) # to keep both legends


handles_points = [handles[4], handles[3], handles[5]]
labels_points = ["Left bounded (start in 1991)", "Complete Career", "Right bounded (end in 2024)"]
plt.legend(handles=handles_points, 
                           labels=labels_points, 
                           markerscale=3, fontsize=13,
                           bbox_to_anchor=(0.8, -0.15),
                           frameon=True, ncol=3)


plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# getting the decades of the players (1990s, 2000s, 2010s)
calibration_players["generation"] = calibration_players["start_year"] - (calibration_players["start_year"] % 10)
#display(calibration_players["generation"].value_counts())

# creation of the labels for the generations
generations_min = calibration_players.groupby("generation")["start_year"].min()
generations_max = calibration_players.groupby("generation")["start_year"].max()

labels = []

for generation in generations_min.index:
    start = generations_min[generation]
    end = generations_max[generation]
    label = f"{generation}s ({start}-{end})" if start != end else f"{generation}s ({start})"
    labels.append(label)

calibration_players["generation_label"] = calibration_players["generation"].replace(generations_min.index, labels)

calibration_players = calibration_players.sort_values("generation")
hue_order = calibration_players["generation_label"].unique()

In [ ]:
calibration_players["generation_label"].value_counts()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of each distribution by generation
sns.kdeplot(data=calibration_players, 
            x="top_strength", 
            hue="generation_label", 
            hue_order=hue_order,
            log_scale=True, fill=False,
            common_norm=False,
            linewidth=4, alpha=1,
            palette="viridis",
            zorder=2)

# plot of the global distribution (all players) if show_global is True
show_global = False

if show_global:
    sns.kdeplot(data=calibration_players, 
                x="top_strength",
                log_scale=True, fill=False,
                common_norm=False,
                linewidth=2.5, alpha=0.9,
                color="red",
                linestyle="--",
                zorder=3)

plt.title("Maximum Zermelo Strength Distribution by Generation", fontsize=25, weight="bold", pad=35)

plt.xlabel(r"Maximum Zermelo Strength ($\pi_{max}$)", fontsize=18)
plt.xticks(fontsize=12)

plt.ylabel("Density", fontsize=18)

# legend
handles = plt.gca().get_lines()
all_labels = list(hue_order)

if show_global:
    all_labels.append("Global Distribution")


legend = plt.legend(handles=handles, title="Career Start Decade", 
                    labels=all_labels, 
                    fontsize=13, title_fontsize=14, 
                    loc="upper center", frameon=False)
plt.setp(legend.get_title(), fontweight='bold')


plt.grid(visible=True, which="major", axis="x", color="gray", linewidth=0.5, alpha=0.5)
plt.grid(visible=True, which="minor", axis="x", color="gray", linestyle=':', linewidth=0.5, alpha=0.3)

sns.despine()
plt.tight_layout()
plt.show()

The KDE plot reveals similarity across all three decades. The distributions share a similar shape, with peaks around the same strengths, followed by a heavy right tail. While the blue curve (2010s) appears a bit shifted and more volatile, this is likely due to more money and opportunities in recent years, leading to more good players. However, the overall shape remain consistent, which supports the stationarity hypothesis.

In [ ]:
stats = calibration_players.groupby("generation_label")["top_strength"].describe()

stats[["count", "mean", "50%", "std", "max"]]

The `mean` and `50%` (median) value is quite stable across decades, with a slight decrease in the 2010s. 

By looking at the `std` column, it suggests that the system is chaotic. It drops significantly from the 2000s to the 2010s (by a factor of 3)! It would mean that the distributions changed a lot, but here it simply means that linear statistics are not sufficient to capture the distribution of strengths. There is a correlation between the maximum strength and the standard deviation!

The standard deviation is dominated by the right tail of the distribution, and is influenced a lot by the presence of a few very strong players. We should move to the logarithmic scale:

In [ ]:
calibration_players["log10_top_strength"] = np.log10(calibration_players["top_strength"])

log_stats = calibration_players.groupby("generation_label")["log10_top_strength"].describe()

log_stats[["count", "mean", "50%", "std", "max"]]

When analysed in the log scale, the standard deviation is much more stable across decades, confirming that the apparent increase in variability was due to the presence of a few outliers in the right tail of the distribution. 

The log transformation helps to stabilize the variance and helps for finding theunderlying distribution of player strengths and confirming that the generation process has remained consistent over time.

<font color="purple"> _Conclusion_: **We can therefore validate the stationary hypothesis of maximum strength distribution**, and pool all players together to estimate the distribution of intrinsic potentials. We will use the logarithm of `top_strength` for the calibration of our model parameters. </font>

In [ ]:
# saving the calibration players data in a .csv

output_calibration_path = f"../data/processed/calibration_players_{calibration_start}-{calibration_end}.csv"
calibration_players.to_csv(output_calibration_path, index=False)

### 1.2. Stationarity of the Number of New Players (Quantity)

To ensure having a realistic simulation, we need to model the arrival of new players. Instead of a fixed number, we could the incoming flux using a Gaussian distribution $\mathcal{N}(\mu, \sigma)$, where $\mu$ is the average number of new players per year and $\sigma$ is the standard deviation of the number of new players per year. The other possibility is a linear regression to see whether there is a significant trend.

In [ ]:
# counting the number of players starting each year in the calibration period
numbers_per_year = calibration_players.groupby("start_year")["start_year"].count()

In [ ]:
# calculating the mean and standard deviation of the number of new players per year
arrival_mean = numbers_per_year.mean()
arrival_std = numbers_per_year.std()

print(f"Average number of new players per year (\u03bc): {arrival_mean:.2f}")
print(f"Standard deviation of new players per year (\u03c3): {arrival_std:.2f}")

In [ ]:
# histogram of the number of new players per year

plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.barplot(x=numbers_per_year.index.values, y=numbers_per_year.values, color="#3498db", 
            alpha=0.6, zorder=2, edgecolor="black")

# horizontal line for the mean value
plt.axhline(y=arrival_mean, 
            color="#e74c3c", label=rf"Mean value: $\mu_A$={arrival_mean:.1f}", 
            zorder=3, linewidth=3, linestyle="-")

# zone containing the mean value +/- 2 standard deviations (95% of the data if normal distribution)
plt.axhspan(ymin=arrival_mean - 2*arrival_std, 
            ymax=arrival_mean + 2*arrival_std, 
            color="#e74c3c", alpha=0.1, label=rf"95% CI ($\mu_A \pm 2\sigma_A$, $\sigma_A = {arrival_std:.1f}$)", zorder=0)

plt.title(f"Flux of New Players Entering the Tour per Year ({calibration_start}-{calibration_end})", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12, ticks=range(0, len(numbers_per_year),3))

plt.ylabel("Number of New Players ($N_{new}$)", fontsize=18)


plt.grid(visible=True, axis="y", linewidth=0.5, alpha=0.7, zorder=0)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

#### H1: Gaussian Distribution Guess

We see that $\sigma$ >> $\sqrt{\mu}$ $(\approx 26.1$), which means that a Poisson distribution would not be appropriate there. To see if the Gaussian distribution is a good fit, we can do a Shapiro-Wilk test (more robust than the Kolmogorov-Smirnov test for small samples) for normality. $\textbf{The null hypothesis is that the data is normally distributed.}$ If the p-value is greater than a significance level (e.g., 0.05), we fail to reject the null hypothesis, suggesting that the data is consistent with a normal distribution. But this test is not sufficient, so we will also look at the histogram and the Q-Q plot of the data to visually assess the normality.

We observed on the previous graph that the shaded area representing the 95% confidence interval (mean ± 2 standard deviations) covers most of the data points, which is consistent with the properties of a normal distribution.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html
from scipy.stats import shapiro

shapiro_stat_normal, shapiro_p_value_normal = shapiro(numbers_per_year)

print(f"Shapiro-Wilk Test Statistic: {shapiro_stat_normal:.4f}")
print(f"Shapiro-Wilk Test p-value: {shapiro_p_value_normal:.4f}")

if shapiro_p_value_normal > 0.05:
    print("\nFail to reject the null hypothesis: the data is consistent with a normal distribution.")
else:
    print("\n Reject the null hypothesis: the data is not consistent with a normal distribution.")

To confirm visually the result of the test (as the number of data points is quite small), we use a Q-Q Plot:
 
- X axis: Theoretical quantiles from  a standard normal distribution $\mathcal{N}(0,1)$, corresponding to a division of the data into $n$ slices of equal probability (where $n$ is the number of data points). For each slice $i$, we calculate $P_i$=$(i-0.5)/n$, which gives us the cumulative probability up to that slice. We then convert it into a Z-score using the probit function (the inverse of the cumulative distribution function): $Z_i$ = $\Phi^{-1}(P_i)$. It representes the theoretical distance from the mean measured in units of standard deviation.


- Y axis: The actual data points, sorted in ascending order. Each point corresponds to the number of new players in a given year, ordered from the smallest to the largest.

- Red line: The line represents the expected relationship if the data were perfectly normally distributed, i.e. Y= $\mu + \sigma X$. If the points closely follow this line, it suggests that the data is consistent with a normal distribution.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html
from scipy.stats import probplot

# osm and osr: tuple of theoretical quantiles and ordered values of the data, used for plotting the Q-Q plot
# slope and intercept and r: standard deviation, mean and correlation coefficient of the data (used for the red line in the Q-Q plot)
(osm_normal, osr_normal), (slope_normal, intercept_normal, r_normal) = probplot(numbers_per_year, dist="norm")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of points in the Q-Q plot
plt.scatter(osm_normal, osr_normal, color="#3498db", edgecolor="black", s=200, zorder=3)

# plot of the red line representing the expected relationship if the data were perfectly normally distributed
x_line = np.array([min(osm_normal), max(osm_normal)])
y_line = intercept_normal + slope_normal * x_line
plt.plot(x_line, y_line, color="#e74c3c", linewidth=3, zorder=2,
         label=r"Normal Reference Line ($Y = \mu_A + \sigma_A X$)")



plt.title("Normal Q-Q Plot of the Number of New Players per Year", fontsize=25, weight="bold", pad=35)
plt.xlabel("Theoretical Quantiles X (Standard Z-scores)", fontsize=18)
plt.ylabel("Data Quantiles Y (Number of New Players)", fontsize=18)


plt.grid(visible=True, axis="x", color="gray", linewidth=0.5, alpha=0.3)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper left")
plt.tight_layout()
plt.show()

This graph shows a strong linear trend. Most of the data points are closely aligned with the red reference line, particularly in the $[-1.5,1.5]$ range. This indicates that the core of the distribution matches the Normal model very well. We observe slight deviations at the extremities, especially for the maximum value. It suggest that these extreme values occur more frequently in reality than a perfect normal distribution would predict.

Finally, we can calculate the coefficient of determination $R^2$, which measures the goodness of fit of our data to the normal data. Here, it is the correlation between the theoretical normal quantiles and the observed quantiles. A value close to 1 indicates that the normal distribution is a highly accurate representation of the data.  

In [ ]:
print(f"R^2 = {r_normal**2:.3f}")

#### H2: Linear Regression Guess

The hypothesis of a normal distribution seems to be okay, but the graph at the beginning of $\textit{Section 1.2}$ suggests that there might be a slight increasing trend in the number of new players over time. So let's try to do a linear regression and get the slope and intercept for the number of new players per year, to see whether it is significant or not.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html
from scipy.stats import linregress

# extract the years and the incoming number of players (as well as coefficient determination R², p-value, and )
years = numbers_per_year.index.values
years_shift = years - calibration_start # so that the regression is done from 0 to len(years) and not from calibration_start to calibration end

incoming_players = numbers_per_year.values

incoming_players_regression = linregress(years_shift,incoming_players)

print(f"Slope: {incoming_players_regression.slope:.2f} ± {incoming_players_regression.stderr:.2f}")
print(f"Intercept: {incoming_players_regression.intercept:.2f} ± {incoming_players_regression.intercept_stderr:.2f}")
print(f"Coefficient of determination: R^2 = {incoming_players_regression.rvalue**2:.2f}")
print(f"p-value: {incoming_players_regression.pvalue:.2e}")

if incoming_players_regression.pvalue < 0.05:
    print("\n There is a significant trend in the number of new players per year!")
else:
    print("\n There is no significant trend in the number of new players per year!")

In [ ]:
# calculating the fitted values for the number of new players per year (based on the linear regression)
fit_new_players = incoming_players_regression.slope*years_shift + incoming_players_regression.intercept

In [ ]:
# histogram of the number of new players per year

plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.barplot(x=numbers_per_year.index.values, y=numbers_per_year.values, color="#3498db", 
            alpha=0.6, edgecolor="black", zorder=1)

plt.plot(range(len(years)), fit_new_players, 
         label=f"Linear Fit: $N_{{new}}(t_0) = {incoming_players_regression.slope:.1f}(t_0 - {calibration_start}) + {incoming_players_regression.intercept:.0f}$", 
         color="#e74c3c", zorder=3, linewidth=5, linestyle="-")


plt.title(f"Flux of New Players Entering the Tour per Year ({calibration_start}-{calibration_end})", fontsize=25, weight="bold", pad=35)

plt.xlabel("Career Start Year ($t_0$)", fontsize=18)
plt.xticks(rotation=45, fontsize=12, ticks=range(0, len(numbers_per_year),3))

plt.ylabel("Number of New Players ($N_{new}$)", fontsize=18)


plt.grid(visible=True, axis="y", linewidth=0.5, alpha=0.7, zorder=0)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

The linear regression confirms a significant upward trend (p-value $<0.05$). The hypothesis of stationarity for the quantity of incoming players is then rejected. However, a linear equation predicts exactly the number of players and lacks the volatility of the real world. To built a realistic simulation, we give some "noise" around this trend. 

We can use a Gaussian distribution with mean 0 and standard deviation equal to the standard deviation of the differences between the real number of incoming players and the predicted number by the regression (residuals). 

Let's see if it is a good choice, by stating that the null hypothesis is that the differences between the real number of incoming players and the predicted number by the regression are normally distributed with mean 0 and standard deviation equal to the standard deviation of the differences. We can use the Shapiro-Wilk test to test this hypothesis, as well as a Q-Q plot.

In [ ]:
# calculate the difference between the real number of incoming players and the predicted number by the regression (residuals)
difference_new_players = fit_new_players - incoming_players

# calculate the mean and standard deviation of the difference

residuals_mean = difference_new_players.mean()
residuals_std = difference_new_players.std()

print(f"Mean of the residuals (difference between real and predicted number of new players): {residuals_mean:.2e}")
print(f"Standard deviation of the residuals: {residuals_std:.2f}")

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html
from scipy.stats import shapiro

shapiro_stat_regression, shapiro_p_value_regression = shapiro(difference_new_players)

print(f"Shapiro-Wilk Test Statistic: {shapiro_stat_regression:.4f}")
print(f"Shapiro-Wilk Test p-value: {shapiro_p_value_regression:.4f}")

if shapiro_p_value_regression > 0.05:
    print("\nFail to reject the null hypothesis: the data is consistent with a normal distribution.")
else:
    print("\n Reject the null hypothesis: the data is not consistent with a normal distribution.")


In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html
from scipy.stats import probplot

# osm and osr: tuple of theoretical quantiles and ordered values of the data, used for plotting the Q-Q plot
# slope and intercept and r: standard deviation, mean and correlation coefficient of the data (used for the red line in the Q-Q plot)
(osm_regression, osr_regression), (slope_regression, 
                                   intercept_regression, r_regression) = probplot(difference_new_players, dist="norm")

In [ ]:
print(arrival_std)

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plot of points in the Q-Q plot
plt.scatter(osm_regression, osr_regression, color="#3498db", edgecolor="black", s=200, zorder=3)

# plot of the red line representing the expected relationship if the data were perfectly normally distributed
x_line = np.array([min(osm_regression), max(osm_regression)])
y_line = intercept_regression + slope_regression * x_line
plt.plot(x_line, y_line, color="#e74c3c", linewidth=3, zorder=2,
         label=r"Normal Reference Line ($Y = 0 + \sigma_{{res}} X$)")



plt.title("Normal Q-Q Plot of the Residuals", fontsize=25, weight="bold", pad=35)
plt.xlabel("Theoretical Quantiles X (Standard Z-scores)", fontsize=18)
plt.ylabel("Data Quantiles Y (Residuals)", fontsize=18)


plt.grid(visible=True, axis="x", color="gray", linewidth=0.5, alpha=0.3)
sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
print(f"R^2 = {r_regression**2:.3f}")

Visually, it shows great alignment with the red theoretical line, and the SW test yields a p-value well above 0.05. It proves that fluctuations could be modeled by a Gaussian distribution!

#### Conclusion for the Generation of New Players each year

Based on this analysis, we now have 2 possible mathematical models to generate the influx of new players $N_{new}(t)$ for year $t$ in the simulation. Note that we can't generate a non-integer number of players, so the final result will be rounded to the nearest integer.


**1. Stationary Model (Gaussian distribution)**

This is the simplest model, assuming the generation is constant over time. The distribution depends only on the average $\mu_A$, with standard deviation $\sigma_A$ (calculated from the available data):

$$N_{new}(t) = \text{Round}(X_t), \quad X_t \sim \mathcal{N}(\mu_A, \sigma_A^2)$$

_(Remark: it ignores the expansion of the number of new players, and understimates the number of players in a long future.)_

**2. Dynamic Model (Linear Regression + "Gaussian Noise")**

This model combines the upward expansion of the tour with normally distributed residuals (noise for more reality). This means that:

$$N_{new}(t) = \text{Round} \left( \alpha_A \cdot (t - t_{ref}) + \beta_A + \epsilon_t \right), \quad \epsilon_t \sim \mathcal{N}(0, \sigma_{res}^2)$$

where $t_{ref}$ the reference year of the calibration (e.g. 1992), $\alpha_A$ and $\beta_A$ respectively the slope (new number of players each year) and intercept (number of new players at $t_{ref}$) of the regression. Here, $\epsilon_t$ is the "noise" added based on the standard deviation $\sigma_{res}$ of the residuals.

#### Saving the parameters

For completeness, we save the parameters of both the stationary model and the dynamic one (with increase of number of players with years) in a $\textit{.json}$ file.

In [ ]:
# saving the arrival parameters (mean and std) in .json file

arrival_data = {
    "arrival_params": {
        "calibration_period" : [calibration_start, calibration_end],

        "stationary_model": {
            "mu": arrival_mean,
            "sigma": arrival_std},

        "dynamic_model": {
            "reference_year": calibration_start,
            "slope": incoming_players_regression.slope,
            "intercept": incoming_players_regression.intercept,
            "sigma_residuals": residuals_std}

                    }
                }
            

config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        params = json.load(f)
else:
    params = {}

params.update(arrival_data)

os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(params, f, indent=3)

print("Both models' parameters have been saved in the .json file!")

**Final choice: As the number of new players has no real influence on the dynamics of the simulation, we will keep it simple by considering the stationary model, with a Gaussian distribution of mean $\mu_A$ and standard deviation $\sigma_A$.**

## 2. Calibrating the Distribution of Intrinsic Potentials (Talent)

As seen in section 1.1, the distribution of players strengths maximum (intrinsic potential $P$) is stationary. We observed that the raw values are highly skewed, but that the transformation $Y=\ln(P)$ follows a "nice-shaped" distribution (distribution that could be recovered by a known distribution).

Let's look at what the distribution $Y$ looks like (using the data of the calibration period):

In [ ]:
log10_potential_mean = calibration_players["log10_top_strength"].mean()
log10_potential_std = calibration_players["log10_top_strength"].std()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_top_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_top_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

# # plot a vertical line for the mean value (logarithm!)
# plt.axvline(log10_potential_mean, color="#e74c3c", linewidth=3, alpha=0.8, linestyle="--", zorder=2, 
#             label=rf"Logarithmic Mean Value: $\mu_{{\log P}} = {log10_potential_mean:.2f}$")

plt.title(r"Distribution of the Logarithm of Player Potentials ($\log_{10} P$)", fontsize=25, weight="bold", pad=35)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

#### H1: Well-known distributions fitting

We will now calibrate this potential distribution by testing different probability density functions on the logarithmic data.

In [ ]:
# select possible distributions
# https://www.itl.nist.gov/div898/handbook/eda/section3/eda36.htm
from scipy import stats

distributions_dict = {"Normal": stats.norm,
                      "Skew Normal": stats.skewnorm,
                      "Laplace": stats.laplace,
                      "Gumbel (right)": stats.gumbel_r,
                      "Logistic": stats.logistic,
                      "Student's t": stats.t}

# calculate the parameters for the fit
#https://docs.scipy.org/doc/scipy-1.17.0/reference/generated/scipy.stats.fit.html
potential_params = {}
for name, distribution in distributions_dict.items():
    params = distribution.fit(calibration_players["log10_top_strength"]) # warning, returns: shapes, then loc and scale!
    potential_params[name] = {"loc": params[-2],
                              "scale": params[-1]}
    if len(params) > 2:
        potential_params[name].update({"shape": list(params[:-2])})

In [ ]:
# --- uncomment the code to save the potential parameters in the .json file ---

# potential_data = {"potential_params": potential_params}

# # saving the parameters in the .json file created earlier
# config_path = "../config/simulation_params.json"

# if os.path.exists(config_path):
#     with open(config_path, "r") as f: # opening the file in read mode
#         config_data = json.load(f)
# else:
#     config_data = {}

# if "potential_params" not in config_data:
#     config_data["potential_params"] = {}

# config_data["potential_params"].update(potential_data["potential_params"])

# os.makedirs(os.path.dirname(config_path), exist_ok=True)


# with open(config_path, "w") as f: # opening the file in writing mode
#     json.dump(config_data, f, indent=3)

In [ ]:
n_cols = 2
n_rows = (len(distributions_dict)+n_cols-1)//n_cols

plt.figure(figsize=(16, n_rows*5))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_top_strength"].min(), 
                    calibration_players["log10_top_strength"].max(), 
                    1000)

for i, (name, distribution) in enumerate(distributions_dict.items()):
    plt.subplot(n_rows,n_cols,i+1)

    # get the parameters of the fit
    loc = potential_params[name]["loc"]
    scale = potential_params[name]["scale"]
    shape = potential_params[name].get("shape", [])

    # get the legend
    hist_legend = "Log-observed Distribution" if i==0 else None
    KDE_legend = "KDE" if i==0 else None
    fit_legend = "Theoretical Curve" if i==0 else None

    # get the probability density function
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html
    # * to unpack the elements in the list shape (if not empty)
    fit = distribution.pdf(x, *shape, loc=loc, scale=scale)

    # plotting the logarithmic distribution
    sns.histplot(calibration_players["log10_top_strength"], color="#bdc3c7", 
             alpha=0.5, stat="density", bins="auto",
             label=hist_legend)


    sns.kdeplot(calibration_players["log10_top_strength"], color="black", 
                alpha=0.4, linewidth=1.5, label=KDE_legend, linestyle=":")

    
    # plotting the theoritical curve (from the distributions chosen)
    plt.plot(x, fit, color="#2c3e50", linewidth=2.5, label=fit_legend)

    plt.title(f"{name} Distribution", fontsize=16, weight="bold")

    if i%n_cols == 0:
        plt.ylabel("Density")
    else:
        plt.ylabel("")
    
    if i+n_cols >= len(distributions_dict):
        plt.xlabel(r"$\log_{10} P$")
    else:
        plt.xlabel("")

    sns.despine()

plt.suptitle(r"Comparison of Theoretical Distributions for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.figlegend(loc="lower center", ncols=3, fontsize=13, frameon=True, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_top_strength"].min(), 
                    calibration_players["log10_top_strength"].max(), 
                    1000)

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_top_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_top_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

skew_normal_params = potential_params["Skew Normal"]

skew_normal_fit = stats.skewnorm.pdf(x, skew_normal_params["shape"], loc=skew_normal_params["loc"], scale=skew_normal_params["scale"])

# plot of the skew normal fit
plt.plot(x, skew_normal_fit, color="#2c3e50", linewidth=4, label="Skew Normal Fit", zorder=2)


plt.suptitle(r"Comparison of Skewed Normal Distribution for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()



We see that the fit are not really good, because some distributions are symmetric (like the normal distribution) while the data is skewed to the right. The best fit here is the skewed normal distribution. But the result is not perfect, so we will try to combine two Gaussian distributions (so 5 parameters: mean and std of each distribution, as well as weight of each distribution) to see if we can get a better fit.

#### H2: Mixture of 2 Distributions

To fit the parameters of a mixture of 2 distributions, we can use the method of maximum likelihood estimation (MLE). The idea is to find the parameters that maximize the likelihood of observing our data given the model. It will be done by minimizing the negative log-likelihood (equivalent to maximizing the likelihood). Let's try this approach for normal and skewed normal distributions.

Mathematicaly, let $y_i=\log_{10}(P_i)$ where $P_i$ is the potential of player $i$. The probability density function (pdf) of the mixture of 2 distributions for a given value $y_i$ is:

$$f(y_i|\theta) = w \cdot f_1(y_i | \theta_1) + (1-w) \cdot f_2(y_i | \theta_2)$$

where $w$ is the weight of the first distribution ($0<w<1$), $f_1$ and $f_2$ are the pdf of the first and second distribution respectively. $\theta_1$ and $\theta_2$ are the parameters of these distributions (mean and standard deviation for normal distributions).

The goal is now to minimize the negative log-likelihood of the observed data given this model. The likelihood function for a set of observed data points $Y = \{y_1, y_2, ..., y_n\}$ is:

$$L(\theta|Y) = \prod_{i=1}^n f(y_i|\theta)$$

Then, the negative log-likelihood is:

$$-\ln L(\theta|Y) = -\sum_{i=1}^n \ln f(y_i|\theta)$$

By minimizing this negative log-likelihood with respect to the parameters $\theta = \{w, \theta_1, \theta_2\}$, we can find the best-fitting parameters for our mixture model.

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
from scipy.optimize import minimize

In [ ]:
# function that returns for all x the value of the pdf of a mixture of 2 distributions with given parameters 
# calculates f(yi|parameters) for all yi in x
def normal_mixture_pdf(x, w, mean1, std1, mean2, std2):
    return w*stats.norm.pdf(x, loc=mean1, scale=std1) + (1-w)*stats.norm.pdf(x, loc=mean2, scale=std2)

def skewed_normal_mixture_pdf(x, w, mean1, std1, shape1, mean2, std2, shape2):
    return w*stats.skewnorm.pdf(x, a=shape1, loc=mean1, scale=std1) + (1-w)*stats.skewnorm.pdf(x, a=shape2, loc=mean2, scale=std2)

# function that calculates the  - log-likelihood of the data given the parameters of the mixture (to minimize it for the fit)
def neg_log_likelihood_normal(params):
    w, mean1, std1, mean2, std2 = params

    if w < 0 or w > 1 or std1 <= 0 or std2 <= 0: # to avoid invalid parameters
        return 1e9 # a very large number to be sure not finding the solution in this case
    
    pdf_values = normal_mixture_pdf(calibration_players["log10_top_strength"], w, mean1, std1, mean2, std2)
    log_likelihood_normal = np.sum(np.log(pdf_values))
    return -log_likelihood_normal

def neg_log_likelihood_skewed_normal(params):
    w, mean1, std1, shape1, mean2, std2, shape2 = params

    if w < 0 or w > 1 or std1 <= 0 or std2 <= 0: # to avoid invalid parameters
        return 1e9 # a very large number to be sure not finding the solution in this case
    

    pdf_values = skewed_normal_mixture_pdf(calibration_players["log10_top_strength"], w, mean1, std1, shape1, mean2, std2, shape2)
    neg_log_likelihood_skewed_normal = np.sum(np.log(pdf_values))
    return -neg_log_likelihood_skewed_normal

# minimization of the negative log-likelihood to find the best parameters for the normal mixture
# values for initial guess (mean initial value a bit left and right from the mean to help finding the 2 components)
mean_log_strength = np.mean(calibration_players["log10_top_strength"])
std_log_strength = np.std(calibration_players["log10_top_strength"])

minimization_normal = minimize(neg_log_likelihood_normal, 
                               [0.5, mean_log_strength-0.5, std_log_strength, mean_log_strength+0.5, std_log_strength],
                               method="Nelder-Mead")

minimization_skewed = minimize(neg_log_likelihood_skewed_normal, 
                               [0.5, mean_log_strength-0.2, std_log_strength, 1, mean_log_strength+0.2, std_log_strength, -1],
                               method="Nelder-Mead")

if (minimization_normal.success == True and minimization_skewed.success == True):
    print("Minimization of the negative log-likelihood for both mixtures was successful!")
else: 
    if minimization_normal.success == False:
        print("Minimization of the negative log-likelihood for the normal mixture failed:", minimization_normal.message)
    if minimization_skewed.success == False:
        print("Minimization of the negative log-likelihood for the skewed normal mixture failed:", minimization_skewed.message)

Let's have a look at a visualisation of the fit of the mixture of 2 Gaussian/skewed normal distributions.

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_top_strength"].min(), 
                    calibration_players["log10_top_strength"].max(), 
                    1000)

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_top_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_top_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

# plot the gaussian mix (individually + together)
w_normal, mean1_normal, std1_normal, mean2_normal, std2_normal = minimization_normal.x


Gaussian_1 = w_normal*stats.norm.pdf(x, loc=mean1_normal, scale=std1_normal)
Gaussian_2 = (1-w_normal)*stats.norm.pdf(x, loc=mean2_normal, scale=std2_normal)
Gaussian_mix = Gaussian_1 + Gaussian_2

plt.plot(x, Gaussian_1, label=f"Gaussian 1 ({w_normal*100:.1f}%)", linewidth=3, linestyle="--", color="#60a215e8", alpha=0.8)
plt.plot(x, Gaussian_2, label=f"Gaussian 2 ({(1-w_normal)*100:.1f}%)", linewidth=3, linestyle="--", color="#1484cfe8", alpha=0.8)
plt.plot(x, Gaussian_mix, label="Gaussian Combination Fit", linewidth=4, color="#2c3e50")


plt.suptitle(r"Comparison of Gaussian Mixture Model for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

# defining the x range for the fit
x = np.linspace(calibration_players["log10_top_strength"].min(), 
                    calibration_players["log10_top_strength"].max(), 
                    1000)

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_top_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_top_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

# plot the gaussian mix (individually + together)
w_skewed, mean1_skewed, std1_skewed, shape1_skewed, mean2_skewed, std2_skewed, shape2_skewed = minimization_skewed.x


skewed_1 = w_skewed*stats.skewnorm.pdf(x, shape1_skewed, loc=mean1_skewed, scale=std1_skewed)
skewed_2 = (1-w_skewed)*stats.skewnorm.pdf(x, shape2_skewed, loc=mean2_skewed, scale=std2_skewed)
skewed_mix = skewed_1 + skewed_2

plt.plot(x, skewed_1, label=f"Skewed Normal 1 ({w_skewed*100:.1f}%)", linewidth=3, linestyle="--", color="#60a215e8", alpha=0.8)
plt.plot(x, skewed_2, label=f"Skewed Normal 2 ({(1-w_skewed)*100:.1f}%)", linewidth=3, linestyle="--", color="#1484cfe8", alpha=0.8)
plt.plot(x, skewed_mix, label="Skewed Normal Combination Fit", linewidth=4, color="#2c3e50")


plt.suptitle(r"Comparison of Skewed Normal Mixture Model for $\mathbf{log_{10} P}$", 
             fontsize=24, weight="bold", y=1.)
plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)

sns.despine()
plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.tight_layout()
plt.show()

The results are quite good, especially for the skewed normal distribution. It's much better than a single distribution, with one distribution fitting the main part in the middle, and the other distribution fitting the right tail.

#### H3: Non-parametric estimation (KDE)

The last possibility is to use directly the Kernel Density Estimation (KDE) to estimate the pdf of the data.

In [ ]:
from scipy.stats import gaussian_kde

kde = gaussian_kde(calibration_players["log10_top_strength"], bw_method="scott")

#### Saving the parameters 

In [ ]:
skewed_single_params = {
    "shape": potential_params["Skew Normal"]["shape"][0],
    "loc": potential_params["Skew Normal"]["loc"],
    "scale": potential_params["Skew Normal"]["scale"]
}

normal_mix_params = {
    "w": w_normal,
    "mean1": mean1_normal,
    "std1": std1_normal,
    "mean2": mean2_normal,
    "std2": std2_normal
}

skewed_mix_params = {
    "w": w_skewed,
    "mean1": mean1_skewed,
    "std1": std1_skewed,
    "shape1": shape1_skewed,
    "mean2": mean2_skewed,
    "std2": std2_skewed,
    "shape2": shape2_skewed
}

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {} 

if "potential_params" not in config_data:
    config_data["potential_params"] = {}

config_data["potential_params"]["skewed_normal_single"] = skewed_single_params
config_data["potential_params"]["normal_mixture"] = normal_mix_params
config_data["potential_params"]["skewed_normal_mixture"] = skewed_mix_params

with open(config_path, "w") as f:
    json.dump(config_data, f, indent=3)

The kde won't be saved directly in the .json file, but will be available by directly using the data in the .csv file and applying the KDE function (`gaussian_KDE`) on it.

To choose the best model, we can look at the best log-likelihood obtained. We should also look at the visual fit of the distribution, to see whether the tails are well captured (especially the one with the best players). Another test is to use the model selection criteria such as the Akaike Information Criterion (AIC), which penalize models with more parameters to avoid overfitting.

THE AIC Score is defined as follows:

$$ AIC = 2k - 2\ln(L) $$

where $k$ is the number of parameters in the model, and $L$ is the maximum value of the likelihood function for the model. A lower AIC score indicates a better fit to the data.

The BIC is another criterion that penalizes models with more parameters more strongly than the AIC:

$$ BIC = k \ln(N) - 2\ln(L) $$

where $N$ is the number of data points.

In [ ]:
# calculate the best log likelihood for all models (to compare them)
# bigger number means better fit
log_likelihood_normal_mix = -minimization_normal.fun
log_likelihood_skewed_mix = -minimization_skewed.fun

pdf_single_skewed = stats.skewnorm.pdf(calibration_players["log10_top_strength"], 
                                       a=skew_normal_params["shape"], 
                                       loc=skew_normal_params["loc"], 
                                       scale=skew_normal_params["scale"])

log_likelihood_single_skewed = np.sum(np.log(pdf_single_skewed))

print("-- Log-Likelihoods of the different models--")
print(f"Single skewed normal: {log_likelihood_single_skewed:.2f}")
print(f"Gaussian mixture: {log_likelihood_normal_mix:.2f}")
print(f"Skewed normal mixture: {log_likelihood_skewed_mix:.2f}")

# calculate the AIC (Akaike Information Criterion) for each model
# allows to compare the models while taking into account the number of parameters (to avoid overfitting)
# lower AIC score means better model
AIC_single_skewed = 2*3 - 2*log_likelihood_single_skewed # 3 parameters for the single skewed normal
AIC_normal_mix = 2*5 - 2*log_likelihood_normal_mix # 5 parameters for the gaussian mixture (w, mean1, std1, mean2, std2)
AIC_skewed_mix = 2*7 - 2*log_likelihood_skewed_mix # 7 parameters for the skewed normal mixture (w, mean1, std1, shape1, mean2, std2, shape2)

print("\n-- AIC scores of the different models--")
print(f"Single skewed normal: {AIC_single_skewed:.2f}")
print(f"Gaussian mixture: {AIC_normal_mix:.2f}")
print(f"Skewed normal mixture: {AIC_skewed_mix:.2f}")


# calculate the BIC (Bayesian Information Criterion) for each model
# stronger penalty for the number of parameters than AIC
# lower BIC score means better model
N = len(calibration_players) 
BIC_single_skewed = 3*np.log(N) - 2*log_likelihood_single_skewed
BIC_normal_mix = 5*np.log(N) - 2*log_likelihood_normal_mix
BIC_skewed_mix = 7*np.log(N) - 2*log_likelihood_skewed_mix

print("\n-- BIC scores of the different models--")
print(f"Single skewed normal: {BIC_single_skewed:.2f}")
print(f"Gaussian mixture: {BIC_normal_mix:.2f}")
print(f"Skewed normal mixture: {BIC_skewed_mix:.2f}")

**The winner seems to be the mixture of 2 skewed normal distributions, with the best log-likelihood, the lowest AIC and BIC scores, and the best visual fit!**

## 3. Calibrating the Aging Curves (Evolution of Strength with Age)

### Fit the aging curve from the data

To model the evolution of players' strength with age, based on its potential (maximum strength reached), we need to calibrate the aging curve. We look at their Zermelo strength ($\pi_{age}$) at each age, and divide it by their maximum strength(potential $P = \max(\pi_{age}$)) to get the relative strength evolution with age. We can then fit a function on this data to get the aging curve.

As seen in Section 1, players with very short careers reiach their "peak" in 0 or 1 year. Therefore, for the calibration of the aging curve, we must restore the `min_active_years=5` filter.

Instead of plotting this ratio directly, we do a logartimitic transformation to it, because the player potential is drawn from the logarithmic space ($\log_{10}(P)$). This allows us to avoid converging back to the potential space, and to directly apply the aging curve in the logarithmic space:

$$\Delta_{age} = \log_{10} \left( \frac{\pi_{age}}{P} \right) = \log_{10}(\pi_{age}) - \log_{10}(P)$$

($\pi_{age}$ is the strength of the player at age $age$, $P$ is the maximum strength of the player), and $\Delta_{age}$ is the value of the aging curve at age $age$)

For simulating matches, we do not even need to get the aging curve in the potential space, as we can directly apply it in the logarithmic space: when simulating a match between player $A$ (at age $a$) and player $B$ (at age $b$), we can calculate their current logarithmic strength $L$ as:

$$L_A^a = \log_{10}(P_A) + \Delta_{a}$$
$$L_B^b = \log_{10}(P_B) + \Delta_{b}$$

($P_A$ and $P_B$ are the potentials of player $A$ and $B$, $\Delta_{a}$ and $\Delta_{b}$ are the values of the aging curve at age $a$ and $b$ respectively)

With the Bradley-Terry model, the probability of player $A$ winning against player $B$ is then:

$$P(A > B) = \frac{\pi_A^a}{\pi_A^a + \pi_B^b} = \frac{10^{L_A^a}}{10^{L_A^a} + 10^{L_B^b}} = \frac{1}{1 + 10^{L_B^b - L_A^a}}$$


This shows that the probability can be calculated directly in the logarithmic space, without needing to convert back to the potential space. This simplifies the model and allows us to work directly with the logarithmic strengths and aging curves.

In [ ]:
aging_curve_players = calibration_players[calibration_players["active_years"]>=min_active_years].copy()

print(f"Number of players in aging curve: {len(aging_curve_players)}")

In [ ]:
# keep only Zermelo's strengths of players in aging_curve_players
zermelo_aging_curve_data = zermelo_strengths_data[zermelo_strengths_data["player_id"].isin(aging_curve_players["player_id"])]

# merge to create the aging curve data
aging_curve_data = pd.merge(zermelo_aging_curve_data,
                            aging_curve_players[["player_id", "birth_year", "log10_top_strength"]],
                            how="left",
                            on="player_id")

aging_curve_data["age"] = aging_curve_data["year"]-aging_curve_data["birth_year"]

# difference between current_strength and the potential (log10_top_strength)
aging_curve_data["log10_strength_diff"] = np.log10(aging_curve_data["zermelo_strength"])-aging_curve_data["log10_top_strength"]

Let's have a look at the shape of the aging curve, by plotting $\Delta_{age}$ as a function of age, with the median curve.

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the median trend line
sns.lineplot(data=aging_curve_data, 
              x="age", y="log10_strength_diff", 
             estimator="median", errorbar=("pi",50), # colored band containing 50% of the points around the median
             color="#e74c3c",linewidth=3,
             label="Median (IQR band)",
             zorder=3)

# plot of the top 10% trend line
sns.lineplot(data=aging_curve_data, 
              x="age", y="log10_strength_diff", 
             estimator=lambda x: np.percentile(x, 90), # 90th percentile = top 10% (value )
             errorbar=None,
             label="Top 10%",
             color="#8e44ad", linewidth=3, linestyle="--", zorder=4)

# plot of the top 10% trend line
sns.lineplot(data=aging_curve_data, 
              x="age", y="log10_strength_diff", 
             estimator=lambda x: np.percentile(x, 10), # 90th percentile = top 10% (value )
             errorbar=None,
             label="Bottom 10%",
             color="#60a215e8", linewidth=3, linestyle="--", zorder=4)


# plot of the individual points
sns.stripplot(data=aging_curve_data, 
              x="age", y="log10_strength_diff", 
              size=2.5, 
              color="#687475", 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)

plt.title("Aging Curve: Evolution of Relative Logarithmic Strength", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age", fontsize=18)
plt.xticks(range(15, 50, 5), rotation=45, fontsize=12)


plt.ylabel(r"Difference from Peak ($\Delta_{age} = \log_{10}(\pi_{age}) - \log_{10}(P)$)", fontsize=18)



plt.legend(frameon=False, fontsize=13, loc="lower right")
plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

We see that the median curve increases from age 15 to around 25, where it reaches its peak and then decreases. The 10% and 90% percentiles shows the same trend.
Data before age 15 is volatile, due to the fact they are not many. And after age 38, the data is also volatile because there are not many players that continue to play at this age.


In [ ]:
age_start = 15
age_end = 38

In [ ]:
aging_curve_data_filtered = aging_curve_data[(aging_curve_data["age"] >= age_start) & (aging_curve_data["age"] <= age_end)].copy()

print("Number of data points removed:", len(aging_curve_data)-len(aging_curve_data_filtered))


We also want to see if the elite players follow the same aging curve as the median trend, or if they have a different aging pattern. To do this, we can plot the aging curves of the top 10% players (in terms of maximum strength) and compare them to the median curve.

In [ ]:
quantile_age = 0.9

elite_min_log10_potential = calibration_players["log10_top_strength"].quantile(quantile_age)
print(f"Minimum log10 potential for elite players (top {(1-quantile_age) * 100:.0f}%): {elite_min_log10_potential:.2f}")

aging_curve_data_filtered["is_elite"] = aging_curve_data_filtered["log10_top_strength"] >= elite_min_log10_potential

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the data by separating elite and non-elite players (with hue)
sns.stripplot(data=aging_curve_data_filtered, 
            x="age", y="log10_strength_diff",
            hue="is_elite", 
            palette=["#6a727340", "#e59718c5"],
            size=3,
            jitter=0.3, 
            native_scale=True, # same scale for both categories
            zorder=2, legend=False)

# plot of the median trend line for all players
sns.lineplot(data=aging_curve_data_filtered, 
            x="age", y="log10_strength_diff", 
            estimator="median", errorbar=None,
            color="black",linewidth=3,
            label="Global Median",
            zorder=3)

# plot of the median trend line for elite players only
sns.lineplot(data=aging_curve_data_filtered[aging_curve_data_filtered["is_elite"]], 
            x="age", y="log10_strength_diff", 
            estimator="median", errorbar=None,
            color="#e59718c5",linewidth=4,
            label=f"Elite Median (Top {(1-quantile_age) * 100:.0f}%)",
            zorder=4)


plt.title("Aging Curve: Evolution of Relative Logarithmic Strength", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age", fontsize=18)
plt.xticks(range(15, 40, 5), rotation=45, fontsize=12)


plt.ylabel(r"Difference from Peak ($\Delta_{age} = \log_{10}(\pi_{age}) - \log_{10}(P)$)", fontsize=18)



plt.legend(frameon=False, fontsize=13, loc="lower right")
plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

By looking at the graph, I don't think it's necessary to have a different aging curve for elite players. The peak seems to be at the same age, and the overall shape of the curve is quite similar. The main difference is that elite players start with a higher difference, because they have a massive amount of development ahead of them to reach the top. The mass players have not such a big potential, so they don't develop as much and start with a smaller difference.
$\newline$ **For a first version of the model, we can consider that all players follow the same aging curve.**

Let's try to fit a function on this data to get the aging curve. We can try a polynom of degree 2 ($ax^2 + bx + c$) or 3 ($ax^3 + bx^2 + cx + d$) to modelize the aging curve, and move it vertically to get the curve touching the 0 at the peak age.

In [ ]:
aging_median = aging_curve_data_filtered.groupby("age")["log10_strength_diff"].median().reset_index()

aging_std = aging_curve_data_filtered.groupby("age")["log10_strength_diff"].agg(["std", "count"]).reset_index()
aging_std["error"] = aging_std["std"]/np.sqrt(aging_std["count"])

# same stats but for elite players only
elite_aging_median = aging_curve_data_filtered[aging_curve_data_filtered["is_elite"]].groupby("age")["log10_strength_diff"].median().reset_index()

elite_aging_std = aging_curve_data_filtered[aging_curve_data_filtered["is_elite"]].groupby("age")["log10_strength_diff"].agg(["std", "count"]).reset_index()
elite_aging_std["error"] = elite_aging_std["std"]/np.sqrt(elite_aging_std["count"])

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
from scipy.optimize import curve_fit

# polynom of degree 2 (with 3 parameters)
def polynom_2(x,a,b,c):
    return a*x**2+b*x+c

# polynom of degree 3 (with 4 parameters)
def polynom_3(x,a,b,c,d):
    return a*x**3+b*x**2+c*x+d

ages = aging_median["age"]
median_diff = aging_median["log10_strength_diff"]

popt_2, pcov_2 = curve_fit(polynom_2, ages, median_diff)
perr_2 = np.sqrt(np.diag(pcov_2))

popt_3, pcov_3 = curve_fit(polynom_3, ages, median_diff)
perr_3 = np.sqrt(np.diag(pcov_3))

print("-- Polynom of degree 2: ax^2 + bx + c ---")
print(f"Parameters: a= {popt_2[0]:.2e} ± {perr_2[0]:.2e} , b= {popt_2[1]:.4f} ± {perr_2[1]:.2e}, c= {popt_2[2]:.4f} ± {perr_2[2]:.2e}")

print("\n-- Polynom of degree 3: ax^3 + bx^2 + cx + d ---")
print(f"Parameters: a= {popt_3[0]:.2e} ± {perr_3[0]:.2e}, b= {popt_3[1]:.2e} ± {perr_3[1]:.2e}, c= {popt_3[2]:.2f} ± {perr_3[2]:.2e}, d= {popt_3[3]:.2f} ± {perr_3[3]:.2e}")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

# plot+scatter of the median trend
sns.scatterplot(x=ages, y=median_diff, 
                color="#e74c3c", edgecolor="black", linewidth=1.5, s=100, alpha=0.8, 
                zorder=2, label="Median Data Points")


plt.errorbar(ages, median_diff, yerr=aging_std["error"], 
             fmt='none', ecolor="#e74c3c", elinewidth=1.5,       
             capsize=4, alpha=0.5, zorder=1)


# # plot of the median trend for elite players only
# sns.scatterplot(x=ages, y=elite_aging_median["log10_strength_diff"], 
#                 color="#e59718c5", edgecolor="black", linewidth=1.5, s=100, alpha=0.8, 
#                 zorder=2, label="Elite Median Data Points")

# plt.errorbar(ages, elite_aging_median["log10_strength_diff"], yerr=elite_aging_std["error"], 
#              fmt='none', ecolor="#e59718c5", elinewidth=1.5,       
#              capsize=4, alpha=0.5, zorder=1)


# plot of the polynom fit (degree 2 and 3)
x_fit = np.linspace(ages.min(), ages.max(), 1000)
plt.plot(x_fit, polynom_2(x_fit, *popt_2), color="#60a215e8", linewidth=2, linestyle="-", label="Quadratic Polynom", zorder=1)
plt.plot(x_fit, polynom_3(x_fit, *popt_3), color="#1484cfe8", linewidth=2, linestyle="-", alpha=0.8,label="Cubic Polynom", zorder=1)

plt.title("Aging Curve: Empirical Data vs Polynomial Models", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age", fontsize=18)
plt.xticks(rotation=45, fontsize=12)


plt.ylabel(r"Difference from Peak ($\Delta_{age} = \log_{10}(\pi_{age}) - \log_{10}(P)$)", fontsize=18)

plt.legend(frameon=False, fontsize=13)
plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

We see there that the fit is quite good, especially for the polynom of degree 3, because it allows asymmetry. Let's look at "extrapolation" of the curve, by plotting it for ages outside the calibration range (from 10 to 50 years old).

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

# plot+scatter of the median trend
sns.scatterplot(x=ages, y=median_diff, 
                color="#e74c3c", edgecolor="black", linewidth=1.5, s=100, alpha=0.8, 
                zorder=2, label="Median Data Points")

plt.errorbar(ages, median_diff, yerr=aging_std["error"], 
             fmt='none', ecolor="#e74c3c", elinewidth=1.5,       
             capsize=4, alpha=0.5, zorder=1)

# plot of the polynom fit (degree 2 and 3)
x_fit = np.linspace(ages.min()-6, ages.max()+11, 1000)
plt.plot(x_fit, polynom_2(x_fit, *popt_2), color="#60a215e8", linewidth=2, linestyle="-", label="Quadratic Polynom", zorder=1)
plt.plot(x_fit, polynom_3(x_fit, *popt_3), color="#1484cfe8", linewidth=2, linestyle="-", alpha=0.8,label="Cubic Polynom", zorder=1)

plt.title("Aging Curve: Empirical Data vs Polynomial Models", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age", fontsize=18)
plt.xticks(rotation=45, fontsize=12)


plt.ylabel(r"Difference from Peak ($\Delta_{age} = \log_{10}(\pi_{age}) - \log_{10}(P)$)", fontsize=18)

plt.legend(frameon=False, fontsize=13)
plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

The only problem of the polynom of degree 3 is that it is becomes flat after age 40, which is not realistic. The polynom of degree 2 is more realistic in this case. What we could do for the retirement is to increase the probability of retirement after a certain age (e.g. 40 years old), to avoid having players that continue to play with a flat aging curve.

We keep the polynom of degree 3 for the simulation! We just need to find the small offset to add to the curve to make it touch 0 at the peak age, we may need it later for the simulation.

In [ ]:
x_points = np.linspace(age_start, age_end, 1000)
y = polynom_3(x_points, *popt_3)

# find the age at which the maximum is reached (peak performance)
offset = np.max(y)
print(offset)

polynom_3_cst = popt_3[3]-offset

In [ ]:
# saving the parameters of the aging curve in the .json file

aging_curve_parameters = {
    "aging_curve_params": {
        "min_active_years": min_active_years,
        "global_model":{
        "age_range": [age_start, age_end],
        "polynom_3": {"params": popt_3.tolist(),
            "offset": offset
        }
    }
}}

# saving the parameters in the .json file created earlier
config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {}

if "aging_curve_params" not in config_data:
    config_data["aging_curve_params"] = {}

config_data["aging_curve_params"].update(aging_curve_parameters["aging_curve_params"])

os.makedirs(os.path.dirname(config_path), exist_ok=True)


with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(config_data, f, indent=3)

### Modeling the fluctuations around the aging curve

While the polynom of degree 3 is a good fit, it is too smooth. Players experience period of good form and bad form that made them deviate from the curve. We also have this momentum effect, where players overperforming the curve one year is likely to carry this form to the next one. To capture this, we model the residuals (deviations from the theoretical curve) using an Autoregressive model of order 1 (AR(1)).

The residual $\epsilon_t$ at age $t$ for a given player is defined as: $\epsilon_t = \Delta_{t} - {\Delta}_{theory}(t)$ (with $\Delta_{t}$ the real difference at age $t$ and ${\Delta}_{theory}(t)$ the value of the aging curve at age $t$).

It means that:

$$\epsilon_t = (\log_{10}(\pi_{age}) - \log_{10}(P)) - (\log_{10}(\pi_{theory}) - \log_{10}(P))= \log_{10}(\pi_{age}) - \log_{10}(\pi_{theory}) = \log_{10}\left(\frac{\pi_{age}}{\pi_{theory}}\right)$$

where $\pi_{theory}$ is the expected strength of the player (predicted by the curve).

In [ ]:
# use of polynom-3 function (polynom_3, x, a, b, c, d)
aging_curve_data_filtered["theory_delta"] = polynom_3(aging_curve_data_filtered["age"], popt_3[0], popt_3[1], popt_3[2], popt_3[3])

# calculate the residual (observed data - theoretical data)
aging_curve_data_filtered["residual"] = aging_curve_data_filtered["log10_strength_diff"]-aging_curve_data_filtered["theory_delta"]

aging_curve_data_filtered["residual"].describe(percentiles = [0.25, 0.5, 0.75, 0.9, 0.95])

Looking at this table, we see that the maximum value is around $0.68$, meaning that a player (at a specific age) performed about $10^{0.68}=4.79$ times better thant what the aging curve expected. On the other end, the mininum value is at about $-2$, indicating a massive drop ($10^{-2}=0.01$) (injury, bad form).

The AR(1) model is defined as: $\epsilon_t = \phi \cdot \epsilon_{t-1} + \eta_t$ (with $\phi$ a coefficient between 0 and 1, coresponding to the part of the residual kept from the previous age and $\eta_t$ a noise with $\eta_t \sim \mathcal{N}(0, \sigma_{\eta}^2)$).

To find these parameters, we need to align the data temporally to compare residual at timte $t$ with residual at time $t-1$.

In [ ]:
# sort the data by player and then by age
aging_curve_data_filtered = aging_curve_data_filtered.sort_values(["player_id", "age"], ascending=True)

# new column where the residual is shifted to the previous year (for first year of each player: Nan value!)
aging_curve_data_filtered["residual_before"] = aging_curve_data_filtered.groupby("player_id")["residual"].shift(1)

# WARNING: we have to verify whether there is really 1 year of difference between residual_before and residual
aging_curve_data_filtered["age_before"] = aging_curve_data_filtered.groupby("player_id")["age"].shift(1)

# drop rows with Nan values, and keep only the two columns
AR1_data = aging_curve_data_filtered.dropna() 

# only keep columns where the difference of ages is 1 (and not more!)
AR1_data = AR1_data[(AR1_data["age"]-AR1_data["age_before"])==1]

AR1_data = AR1_data[["residual_before", "residual"]]

Now we can extract $\phi$ and then isolate the noise $\eta_t$ to calculate its standard deviation $\sigma_{\eta}$, from a linear regression!

In [ ]:
from scipy.stats import linregress

# linear regression between the residual before and current one
linregress_AR1 = linregress(AR1_data["residual_before"], AR1_data["residual"])

phi = linregress_AR1.slope
stderr = linregress_AR1.stderr
p_value = linregress_AR1.pvalue
r2_coefficient = linregress_AR1.rvalue**2

print(f"Phi (Slope): {phi:.2f}")
print(f"Standard Error: {stderr:.2e}")
print(f"Coefficient of Determination R^2: {r2_coefficient:.2f}")

In [ ]:
#calculate the std of the random noise (normal distribution) by isolating it from the formula

AR1_data["noise"] = AR1_data["residual"]-phi*AR1_data["residual_before"]

sigma_eta = AR1_data["noise"].std()
print(f"Standard Deviation (σ_η): {sigma_eta:.2f}")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

# scatter of all points
sns.scatterplot(data=AR1_data, x="residual_before", y="residual", 
                color="#3498db", s=10, alpha=0.3,
                zorder=1, legend=False)

# plot of the linear regression
x_line = np.array([AR1_data["residual_before"].min(), AR1_data["residual_before"].max()])
plt.plot(x_line, phi * x_line + linregress_AR1.intercept, color="#e74c3c", linewidth=4, label=rf"AR(1) Trend ($\phi$={phi:.2f})", zorder=2)

plt.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.axvline(0, color="gray", linestyle="--", alpha=0.5)


plt.title("Correlation of Player Form", fontsize=25, weight="bold", pad=20)
plt.xlabel(r"Residual at year $t-1$ ($\epsilon_{t-1}$)", fontsize=18)
plt.ylabel(r"Residual at year $t$ ($\epsilon_t$)", fontsize=18)

plt.legend(frameon=False, fontsize=13, loc="upper left")
plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

To check whether the AR(1) model captures the variance of the real data, we can compare the actual std of the residuals with the theoretical std of an AR(1) process, given by 

$\sigma_{theory} = \frac{\sigma_{\eta}}{\sqrt{1 - \phi^2}}$

In [ ]:
AR1_theory_std = sigma_eta / np.sqrt(1-phi**2)
real_std = AR1_data["residual"].std()

print(f"Relative error: {(abs(AR1_theory_std - real_std) / real_std * 100):.2f} %")

Let's see now if the fluctuations are stationary, by looking at which point the fluctuations diverge over 20 years (= career length):

$$ \sigma^2\left(\epsilon_t\right)=\phi^2\sigma^2\left(\epsilon_{t-1}\right)+\sigma^2\left(\eta_t\right) $$

In [ ]:
career_length = 20

var_eta = sigma_eta**2

variance_history = []

previous_residual_variance = 0

for i in range(career_length):
    residual_variance = (phi**2)* previous_residual_variance + var_eta
    variance_history.append(residual_variance)
    previous_residual_variance = residual_variance


plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


sns.scatterplot(x=np.arange(0, career_length, 1), y=variance_history, 
                color="#e74c3c", edgecolor="black", linewidth=1.5, s=100, alpha=0.8, 
                zorder=2)
plt.plot(np.arange(0, career_length, 1),variance_history, color="black", linewidth=1, linestyle="-", zorder=1)


plt.title("Evolution of Fluctuations Variance over a 20-Year Career", fontsize=25, weight="bold", pad=20)
plt.xlabel(r"Step", fontsize=18)
plt.xticks(np.arange(0,career_length,2))
plt.ylabel(r"Variance ($\sigma^2$)", fontsize=18)

plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fluctuations_params = {
    "phi": phi,
    "sigma_eta": sigma_eta
}

# saving the parameters in the .json file created earlier
config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {}

if "aging_curve_params" not in config_data:
    config_data["aging_curve_params"] = {}

if "global_model" not in config_data["aging_curve_params"]:
    config_data["aging_curve_params"]["global_model"] = {}

config_data["aging_curve_params"]["global_model"]["fluctuations_params"] = fluctuations_params

os.makedirs(os.path.dirname(config_path), exist_ok=True)


with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(config_data, f, indent=3)

**Choices for the simulation**:

- Althoug elite players start with a higher difference to peak (more potential to reach), we will use a single curve for all players (for simplicity). We will come back to this later if needed.

- In the simulation, the relative strength calculated with the aging curve and with added flucutations imply that it is possible for a player to exceed their potential $P$ at their peak age. This doesn't seem to be a huge problem.

- Our first simulation will turn off the fluctuations. This allows us to observe identical career paths for players born with the exact same potential. We will come back to this later if necessary.

### New method: Stratifying Players into 3 Different Groups

The idea there is to have three groups defined from the log10 potential. We have the bottom players, the middle ones and the elites (top). And we want to have an aging curve for each category.

In [ ]:
# define the quantiles to distinguish the three groups
least_quantile = 0.5
top_quantile = 0.9

In [ ]:
# strength limits (in log10) 
lower_limit = calibration_players["log10_top_strength"].quantile(least_quantile)
upper_limit = calibration_players["log10_top_strength"].quantile(top_quantile)

print(rf"Lower Limit: {lower_limit:.2f}")
print(f"Upper Limit: {upper_limit:.2f}")

In [ ]:
# attributing each player a category (bottom, middle, top)
aging_curve_conditions=[(aging_curve_data_filtered["log10_top_strength"] < lower_limit), 
                        (aging_curve_data_filtered["log10_top_strength"] >= lower_limit) & 
                        (aging_curve_data_filtered["log10_top_strength"] < upper_limit),
                        (aging_curve_data_filtered["log10_top_strength"] >= upper_limit)]

aging_curve_choices = ["bottom", "middle", "top"]

aging_curve_data_filtered["category"] = np.select(aging_curve_conditions, aging_curve_choices, default="unknown")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})


# plot of the data by separating elite and non-elite players (with hue)
sns.stripplot(data=aging_curve_data_filtered, 
            x="age", y="log10_strength_diff",
            hue="category", 
            palette=["#1484cfe8","#60a215e8", "#e59718c5"],
            size=3,
            jitter=0.3, alpha=0.3,
            native_scale=True, # same scale for both categories
            zorder=2, legend=False)

# plot of the median trend line for all players
sns.lineplot(data=aging_curve_data_filtered, 
            x="age", y="log10_strength_diff",
            hue="category", 
            estimator="median", errorbar=None,
            palette=["#1484cfe8","#60a215ff", "#e59718c5"],
            linewidth=4,
            zorder=3)

plt.title("Aging Curve: Evolution of Relative Logarithmic Strength", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age", fontsize=18)
plt.xticks(range(15, 40, 5), rotation=45, fontsize=12)


plt.ylabel(r"Difference from Peak ($\Delta_{age} = \log_{10}(\pi_{age}) - \log_{10}(P)$)", fontsize=18)
plt.ylim(-2,0)


plt.legend(frameon=False, fontsize=13, loc="lower center")
plt.grid(visible=True, which="major",  axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# get the median points for each age and categories
aging_categories_median = aging_curve_data_filtered.groupby(["category", "age"])["log10_strength_diff"].median().reset_index()

aging_categories_std = aging_curve_data_filtered.groupby(["category", "age"])["log10_strength_diff"].agg(["std", "count"]).reset_index()
aging_categories_std["error"] = aging_categories_std["std"]/np.sqrt(aging_categories_std["count"])

In [ ]:
# get the number of bottom, middle and top players for each age, plot it as a function of age

# group by age and category to count the number of players
player_counts = aging_curve_data_filtered.groupby(['age', 'category']).size().reset_index(name='count')

# plot the number of players per category as a function of age
plt.figure(figsize=(16, 9))
sns.set_style("ticks")
plt.rcParams.update({'font.family': 'serif'})

sns.lineplot(data=player_counts, x='age', y='count', hue='category', 
             palette=["#e59718c5","#60a215e8","#1484cfe8", ], linewidth=3, marker='o')

plt.title("Number of Players per Category as a Function of Age", fontsize=25, weight="bold", pad=35)
plt.xlabel("Player Age", fontsize=18)
plt.xticks(range(15, 40, 5), rotation=45, fontsize=12)
plt.ylabel("Number of Players", fontsize=18)

plt.legend(frameon=False, fontsize=13, loc="upper right")
plt.grid(visible=True, which="major", axis="x", linewidth=0.5, alpha=0.7)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fit_categories_ages = {
    "bottom": (18, 29),
    "middle": (17,33),
    "top": (15, 38)
}

In [ ]:
stratified_aging_params = {}

plt.figure(figsize=(16, 18))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

for i, category in enumerate(aging_categories_median["category"].unique()):

    plt.subplot(3, 1, i+1)


    # getting the fit age ranges
    age_min_fit, age_max_fit = fit_categories_ages[category]

    # filtering the data by category (only fit data) 
    category_data = aging_categories_median[aging_categories_median["category"] == category]
    data_to_fit = category_data[(category_data["age"]>=age_min_fit) & (category_data["age"]<=age_max_fit)]
    ages_to_fit = data_to_fit["age"]
    diffs_to_fit = data_to_fit["log10_strength_diff"]

    # full data
    ages_all = category_data["age"]
    diffs_all = category_data["log10_strength_diff"]

    # getting the std 
    category_std_data = aging_categories_std[aging_categories_std["category"] == category]
    categories_errors = category_std_data["error"]
    errors_to_fit = category_std_data[(category_std_data["age"]>=age_min_fit) & 
                                      (category_std_data["age"]<=age_max_fit)]["error"]

    # fit of the data
    popt_2_categories, pcov_2_categories = curve_fit(polynom_2, ages_to_fit, diffs_to_fit)
    popt_3_categories, pcov_3_categories = curve_fit(polynom_3, ages_to_fit, diffs_to_fit)

    # finding the offsets
    x_points = np.linspace(ages_all.min(), ages_all.max(), 1000)
    y_points_2 = polynom_2(x_points, *popt_2_categories)
    y_points_3 = polynom_3(x_points, *popt_3_categories)

    x_fit_points = np.linspace(ages_to_fit.min(), ages_to_fit.max())
    y_fit_points_2 = polynom_2(x_fit_points, *popt_2_categories)
    y_fit_points_3 = polynom_3(x_fit_points, *popt_3_categories)
    
    offset_2 = np.max(y_fit_points_2)
    offset_3 = np.max(y_fit_points_3)

    # save all parameters in the dictionary
    stratified_aging_params[category] = {
        "age_range": [age_min_fit, age_max_fit],
        "polynom_2": {
            "params": popt_2_categories.tolist(),
            "offset": offset_2},
        "polynom_3": {
            "params": popt_3_categories.tolist(), 
            "offset": offset_3
        }
    }

    # get the constant value for the fixed value of bottom players
    if category == "bottom":
        stratified_aging_params[category]["constant"] = {"params": np.mean(diffs_to_fit),
                                                         "offset": np.mean(diffs_to_fit)}
    # plot of the data
    sns.scatterplot(x=ages_all, y=diffs_all, 
                    color="gray", edgecolor="black", linewidth=1.5, s=100, alpha=0.8, 
                    zorder=2, label="All Data Points")

    plt.errorbar(ages_all, diffs_all, yerr=categories_errors, 
                 fmt='none', ecolor="gray", elinewidth=1.5,       
                 capsize=4, alpha=0.5, zorder=1)

    
    sns.scatterplot(x=ages_to_fit, y=diffs_to_fit, 
                    color="#e74c3c", edgecolor="black", linewidth=1.5, s=100, alpha=0.8, 
                    zorder=3, label="Fitted Data Points")

   
    plt.errorbar(ages_to_fit, diffs_to_fit, yerr=errors_to_fit, 
                 fmt='none', ecolor="#e74c3c", elinewidth=1.5,       
                 capsize=4, alpha=0.5, zorder=4)

    plt.plot(x_points, y_points_2, color="#60a215e8", linewidth=2, linestyle="-", label="Quadratic Polynom", zorder=1)
    plt.plot(x_points, y_points_3, color="#1484cfe8", linewidth=2, linestyle="-", alpha=0.8,label="Cubic Polynom", zorder=1)

    plt.title(f"Aging Curve: {category} players", fontsize=20, weight="bold", pad=15)
    plt.xlabel("Player Age", fontsize=16)
    plt.ylabel(r"$\Delta_{age}$", fontsize=16)
    plt.legend(frameon=False, fontsize=13)
    plt.grid(visible=True, which="major", axis="x", linewidth=0.5, alpha=0.7)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# get the coefficients of the aging curve for the bottom players
stratified_aging_params["bottom"]["polynom_2"]["params"]

We have to keep the best fit for the model:
- Top Players: the best curve seems to be the cubic polynom.
- Middle Players: The quadratic curve is fine.
- Bottom: Quadratic and Cubic polynoms are not good candidates, it seems to be constant (the slope of the linear regression is really small). We will assume here that the aging curve is constant. 

In [ ]:
# saving the parameters in the .json file created earlier

stratify_model_params = {
        "categories_limit": [lower_limit, upper_limit],

        "bottom": {
            "ages_range": stratified_aging_params["bottom"]["age_range"],
            "constant": stratified_aging_params["bottom"]["constant"]
        },

        "middle": {
            "ages_range": stratified_aging_params["middle"]["age_range"],
            "polynom_2": stratified_aging_params["middle"]["polynom_2"]
        },

         "top": {
            "ages_range": stratified_aging_params["top"]["age_range"],
            "polynom_3": stratified_aging_params["top"]["polynom_3"]
        }
}

config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {}

if "aging_curve_params" not in config_data:
    config_data["aging_curve_params"] = {}


config_data["aging_curve_params"]["stratified_model"] = stratify_model_params

os.makedirs(os.path.dirname(config_path), exist_ok=True)


with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(config_data, f, indent=3)

## 4. Calibrating Career Duration and Retirement

### 4.1. Entry Age

Let's see first the distribution of starting age, and whether there is a correlation between the potential $log_{10}(P)$ with the starting age:

In [ ]:
start_age_min = calibration_players["start_age"].min()
start_age_max = calibration_players["start_age"].max()


plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

sns.histplot(calibration_players["start_age"], 
             color="#bdc3c7", 
             alpha=0.8, edgecolor="white", stat="density", 
             bins=np.arange(start_age_min - 0.5, start_age_max + 1.5, 1), zorder=1,
             label="Observed Distribution")

sns.kdeplot(calibration_players["start_age"], color="black", alpha=0.4, linewidth=2.5, zorder=3, label="KDE", linestyle = ":")

plt.title("Distribution of the Starting Age", fontsize=25, weight="bold", pad=20)
plt.xlabel("Starting Age", fontsize=18)
plt.ylabel("Density", fontsize=18)

plt.legend(frameon=False, fontsize=13, loc="upper center")
plt.grid(visible=True, which="major", axis="x", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

sns.stripplot(data=calibration_players, x="start_age", y="log10_top_strength",
              size=3, 
              color="#687475", 
              jitter=0.3, #to avoid overplotting
              native_scale=True, 
              zorder=2)


plt.title(r"Logarithm of Player Potentials ($\log_{10} P$) vs. Starting Age", fontsize=25, weight="bold", pad=35)


plt.title(r"Logarithm of Player Potentials ($\log_{10} P$) ", fontsize=25, weight="bold", pad=20)
plt.xlabel("Starting Age", fontsize=18)
plt.ylabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)

plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

start_age_corr, p_start_age_corr = pearsonr(calibration_players["start_age"], calibration_players["log10_top_strength"])
print(f"Correlation Coefficient (Pearson): {start_age_corr:.2f}")
print(f"P-value: {p_start_age_corr:.2e}")

The scatter plot and correlation coefficient reveals a significant but weak negative correlation between a player's maximum potential and their starting age. While it makes sense that top-talented players tend to enter the tour earlier than average players, the relationship is too weak to justifying building a complex dependance between $P$ and the entry age in our model. 

As the entry age has no impact on dynamics of the rankings systems we aim to evaluate, we do not take into account the correlation between these 2 variables to avoid unnecessary complexity. 

**First decision**: Fix the starting age for all generated players to a constant value corresponding to the empirical mean. 

In [ ]:
calibration_players["start_age"].describe(percentiles = [0.25, 0.5, 0.75, 0.9, 0.95])

The median value shown (18 years old) is used because, unlike the mean value, it is not influenced by the maximum value (62 years old, which is unrealistic...). It can therefore be saved in the .json file:

In [ ]:
fixed_start_age = int(calibration_players["start_age"].median())

As seen in the 3rd notebook, the distribution of the retirement ages for the synthetic data doesn't match the observed one. This is explained by the fact that the starting age was fixed at the median value, implying that most bottom players are going to retire at age 18. For dealing with this problem, we can change the starting age by considering different possible values. We see with the histogram that the most likely ages of entry are between 16 and 29. And it has no sens for a player to enter the tournaments at the age of 30 for example, because he would already be declining (aging curve is declining at this age). For these reasons, we will randomly select an entry age between 16 and 20, taking into account the associated probabilities (by removing all data points with starting ages outside this range).

In [ ]:
min_age_model = 15
max_age_model = 29

start_age_model_players = calibration_players[(calibration_players["start_age"]>=min_age_model) &
                                        (calibration_players["start_age"]<=max_age_model)]

proportion_start_age_players = np.round(len(start_age_model_players)/len(calibration_players)*100,2)

print(f"Proportion of players with a start age between {min_age_model} and {max_age_model}: {proportion_start_age_players}%")

In [ ]:
start_age_probabilities=start_age_model_players["start_age"].value_counts(normalize=True).sort_index().reset_index()
start_age_probabilities.columns = ["start_age", "probability"]

In [ ]:
start_age_probabilities

In [ ]:
# saving the parameters in the .json file created earlier
config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {}

if "entry_age_params" not in config_data:
    config_data["entry_age_params"] = {}

# save the probabilities of each start age
distribution_start_ages = {"ages": start_age_probabilities["start_age"].tolist(),
                           "probabilities": start_age_probabilities["probability"].tolist()}

config_data["entry_age_params"]["fixed_start_age"] = fixed_start_age
config_data["entry_age_params"]["distribution_ages"] = distribution_start_ages 

os.makedirs(os.path.dirname(config_path), exist_ok=True)


with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(config_data, f, indent=3)

### 4.2. Retirement Process

To simulate when players leave the tour, we must calculate the probability of retirement at each age. 

Here, we must pay attention to the career gaps some players have. Let's look at some stats about it:

In [ ]:
# gap years = (end year - start year) - active years
# do not forget to add +1!
calibration_players["gap_years"] = (calibration_players["end_year"]-calibration_players["start_year"]+1) - calibration_players["active_years"]

players_with_gaps = calibration_players[calibration_players["gap_years"]>0]

# ratio: players with gaps / total number of players
ratio_gap_players = len(players_with_gaps) / len(calibration_players)

print(f"Number of players with a least 1 missing year: {ratio_gap_players*100:.2f}%")
print(f"Average number of missing years: {players_with_gaps["gap_years"].mean():.2f}")

players_with_gaps["gap_years"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

About 29% of players have at least one gap year in their career (missing data, injury, break from sport, ...). We will assume here that **a player's career is continuous from the start age to the end age**. We will consider in the retirement probability that even if a player do not record a match in a given year, he is sill considred as an active player!

#### Global Retirement Probability (All players considered)
We first analyse the whole dataset with all players, by extracting the retirement probability at each age (ratio between the number of players retiring and total number of active players at this age).

In [ ]:
# function that gets the probability of retirement for each age based on a dataset of players
def get_retirement_prob(players):

    # number of retirements for each age
    retire_per_age = players.groupby("end_age")["player_id"].count()

    #calculate the number of active players at each age
    min_age = players["start_age"].min()
    max_age = players["end_age"].max()

    prob_retire_per_age = {}
    active_per_age = {}

    for age in range(min_age, max_age+1):
        # to be active at age: start age <= age <= end_age
        active_players = players[(players["start_age"] <= age) 
                                            & (players["end_age"] >= age)]
        
        nbr_active_players = len(active_players)
        active_per_age[age] = nbr_active_players

        if len(active_players) > 0:
            prob_retire_per_age[age] = retire_per_age.get(age,0) / len(active_players)
        
        else: prob_retire_per_age[age] = 0

    # dict into pd series
    prob_retire_per_age = pd.Series(prob_retire_per_age)
    active_per_age = pd.Series(active_per_age) 

    return prob_retire_per_age, active_per_age

In [ ]:
# function that plots the data (probabilty of retirement per age, and histogram of active players per age)
def plot_retirement_probabilities(prob_players_retire, active_players, min_age=14, max_age=62, legend_name="All Players"):

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]}, sharex=True) 
    sns.set_style("ticks") 
    plt.rcParams.update({'font.family': 'serif'})

    # filtering the data to keep only the intervall [min_age, max_age]
    prob_players_retire = prob_players_retire.loc[min_age:max_age]
    active_players = active_players.loc[min_age:max_age]


    # plot of the probability of retirement per age
    plt.sca(ax1)
    sns.scatterplot(x=prob_players_retire.index, y=prob_players_retire.values, 
                    color="#e74c3c", edgecolor="black", 
                    linewidth=1.5, s=100, alpha=0.8, 
                    zorder=2)

    plt.plot(prob_players_retire.index, prob_players_retire.values, color="#e74c3c", linewidth=2, zorder=1)

    plt.title("Observed Probability of Retirement by Age", fontsize=16, weight="bold", pad=35)

    plt.ylabel("Probability to retire")
    plt.grid(visible=True, which="major",  axis="both", linewidth=0.5, alpha=0.7)


    # plot of the number of active players per age
    plt.sca(ax2)

    plt.bar(active_players.index, active_players.values, color="#bdc3c7")


    plt.title("Number of active Players by Age", fontsize=16, weight="bold", pad=35)

    plt.xlabel("Player Age",fontsize=17)
    plt.xticks(np.arange(min_age, max_age+1, 5), rotation=45, fontsize=12)
    plt.ylabel("Number of Active Players")
    plt.yscale("log")
    plt.grid(visible=True, which="major",  axis="both", linewidth=0.5, alpha=0.7)


    plt.suptitle(f"Retirement Analysis: Probability and Data available per Age \n ({legend_name})", fontsize=24, weight="bold", y=1.)
    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
prob_all_players_retire, active_all_players = get_retirement_prob(calibration_players)

In [ ]:
plot_retirement_probabilities(prob_all_players_retire, active_all_players, min_age=18, max_age=40)

This graph shows that the probability of retirement increases with age after 18. To model this in our simulation, we need to test functions for fitting the data: linear, quadratic and exponential function.

In [ ]:
# possible fits
# trick by translating the function such that x=0 corresponds to 18yo. (= entry age)

def linear_model(x, a, b):
    return a*(x-18) + b

def quadratic_model(x, a, b, c):
    return a*((x-18)**2) + b*(x-18) + c

def exp_model(x, a, b, c):
    return a * np.exp(b*(x-18))+c 


# function that try to fit functions to the data of prob. of retirement per age
def fit_retirement_data(probabilities, min_age_fit=16, max_age_fit=39):

    ages_fit = np.arange(min_age_fit, max_age_fit+1, 1)
    probs_fit = probabilities[(probabilities.index >= min_age_fit) 
                                & (probabilities.index <= max_age_fit)].values    
    
    #fit
    popt_lin, pcov_lin = curve_fit(linear_model, ages_fit, probs_fit)
    popt_quad, pcov_quad = curve_fit(quadratic_model, ages_fit, probs_fit)

    # try to fit the exponential (if doesn't always work)
    try: 
        popt_exp, pcov_exp = curve_fit(exp_model, ages_fit, probs_fit)
        perr_exp = np.sqrt(np.diag(pcov_exp))

        exp_results = {
            "params": popt_exp,
            "errors": perr_exp
            }
       
    except RuntimeError:
        print("Warning: The exponential fit didn't converge to a solution. Results ignored.")

        exp_results = {
            "params": None,
            "errors": None
            }


    perr_lin = np.sqrt(np.diag(pcov_lin))
    perr_quad = np.sqrt(np.diag(pcov_quad))

    # end product: dictionary containing the parameters of the fit and the errors
    fits_results = {
        "linear": {
            "params": popt_lin,
            "errors": perr_lin
        },
        "quadratic": {
            "params": popt_quad,
            "errors": perr_quad
        },
        "exp": exp_results
    }

    return fits_results

In [ ]:
min_age_fit_all = 16
max_age_fit_all = 39

In [ ]:
fits_results_all = fit_retirement_data(prob_all_players_retire, min_age_fit=min_age_fit_all, max_age_fit=max_age_fit_all)

In [ ]:
# get the parameters for each function
popt_lin_all = fits_results_all["linear"]["params"]
popt_quad_all = fits_results_all["quadratic"]["params"]
popt_exp_all = fits_results_all["exp"]["params"]

In [ ]:
# function that plots the fits with the observed data

def plot_fit_comparisons(prob_players_retire , min_age_fit, max_age_fit, popt_lin, popt_quad, popt_exp, legend_name="All Players"):

    plt.figure(figsize=(16, 9))
    sns.set_style("ticks") 
    plt.rcParams.update({'font.family': 'serif'})

    #plot of the data for each age
    sns.scatterplot(x=prob_players_retire.index, y=prob_players_retire.values, 
                    color="#bdc3c7", edgecolor=None, 
                    linewidth=1.5, s=100, alpha=0.8, 
                    zorder=2, label=f"All Observed Data")

    sns.scatterplot(x=np.arange(min_age_fit, max_age_fit+1), 
                    y=prob_players_retire.loc[min_age_fit:max_age_fit].values, 
                    color="#e74c3c", edgecolor="black", 
                    linewidth=1.5, s=100, alpha=0.9, 
                    zorder=3, label=f"Training Data ({min_age_fit}-{max_age_fit})")

    # models
    ages_plot = np.linspace(prob_players_retire.index.min(), prob_players_retire.index.max())
    plt.plot(ages_plot, linear_model(ages_plot, *popt_lin), label="Linear Regression", color="#1484cfe8", zorder=2)
    plt.plot(ages_plot, quadratic_model(ages_plot, *popt_quad), label="Quadratic Polynom", color="#60a215e8", zorder=2)

    if popt_exp is not None: # verify that the exp. fit worked
        plt.plot(ages_plot, exp_model(ages_plot, *popt_exp), label="Exponential Function", color="#e59718c5", zorder=2)


    plt.title(f"Retirement Probability: Empirical Data vs Numerical Models \n ({legend_name})", fontsize=25, weight="bold", pad=35)

    plt.xlabel("Starting Age", fontsize=18)
    plt.ylabel("Probability to retire", fontsize=18)
    plt.ylim(-0.05,1.05)

    plt.legend(frameon=False, fontsize=13, loc="upper center")
    plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_fit_comparisons(prob_all_players_retire, min_age_fit_all, max_age_fit_all, popt_lin=popt_lin_all, popt_quad=popt_quad_all, popt_exp=popt_exp_all)

The three models are really similar from 18 to 35 years old. The difference after this age is that the exponential function increases faster than the quadratic polynom and the linear fit. For simplicity, we could keep the straight line. Let's save the parameters of the three curves just in case. 

In [ ]:
retirement_params = {

    "global_model": {
        "calibration_ages": [min_age_fit_all, max_age_fit_all],
        "linear": {
            "params": popt_lin_all.tolist(),
            "equation": "p[0]*(age-18) + p[1]" 
        },
        "quadratic": {
            "params": popt_quad_all.tolist(),
            "equation": "p[0]*(age-18)^2 + p[1]*(age-18) + p[2]"
        },
        "exp": {
            "params": popt_exp_all.tolist(),
            "equation": "p[0]*exp(p[1]*(age-18)) + p[2]"
        } 
    }
}

# saving the parameters in the .json file created earlier
config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {}

config_data["retirement_params"] = retirement_params

os.makedirs(os.path.dirname(config_path), exist_ok=True)


with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(config_data, f, indent=3)

#### Stratification of players into 3 groups!
In the previous model, we assumed a single curve for the retirement of all players. However, in reality, career length can be influenced by success (financial and athletic). Elite players tend to playlonger. Conversely, low-ranked players often retire earlier. To test this hypothesis, we will stratify our set of players into three groups basedon their maximum potential ($\log_{10} P$), as we did for the aging curves.

In [ ]:
# get the three groups from the strengths limits
bottom_players = calibration_players[calibration_players["log10_top_strength"] < lower_limit]
middle_players = calibration_players[(calibration_players["log10_top_strength"] >= lower_limit) 
                                     & (calibration_players["log10_top_strength"] < upper_limit)]
top_players = calibration_players[calibration_players["log10_top_strength"] >= upper_limit]

In [ ]:
# visualisation of the three groups on the histogram of strengths

plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

# plotting the logarithmic distribution
sns.histplot(calibration_players["log10_top_strength"], color="#bdc3c7", 
             alpha=0.5, edgecolor="white", stat="density", bins="auto", zorder=1,
             label="Log-observed Distribution")
sns.kdeplot(calibration_players["log10_top_strength"], color="black", alpha=0.4, linewidth=2.5, zorder=2, label="KDE", linestyle = ":")

# plot of the 3 different groups of players
plt.axvline(lower_limit, color="black",alpha=0.7, linewidth=2, linestyle="--", zorder=3)
plt.axvline(upper_limit, color="black", alpha=0.7, linewidth=2, linestyle="--", zorder=3)

plt.axvspan(calibration_players["log10_top_strength"].min(), lower_limit, 
            color="#1484cfe8", zorder=0, alpha=0.2,
            label=f"Group 1 ({least_quantile*100:.0f}%)")
plt.axvspan(lower_limit, upper_limit, 
            color="#60a215e8", zorder=0, alpha=0.2,
            label=f"Group 2 ({(top_quantile-least_quantile)*100:.0f}%)")
plt.axvspan(upper_limit, calibration_players["log10_top_strength"].max(),
            color="#e59718c5", zorder=0, alpha=0.2,
            label=f"Group 3 ({(1-top_quantile)*100:.0f}%)")

plt.title(r"Stratification of Player Potentials for Retirement Process ($\log_{10} P$)", fontsize=25, weight="bold", pad=35)

plt.xlabel(r"Logarithm of Potential ($\log_{10} P$)", fontsize=18)
plt.ylabel("Density", fontsize=18)
plt.legend(frameon=True, fontsize=13, loc="upper right")
plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
prob_bottom_players_retire, active_bottom_players = get_retirement_prob(bottom_players)
prob_middle_players_retire, active_middle_players = get_retirement_prob(middle_players)
prob_top_players_retire, active_top_players = get_retirement_prob(top_players)

In [ ]:
plot_retirement_probabilities(prob_bottom_players_retire, active_bottom_players, min_age=18, max_age=40, legend_name="Bottom Players")
plot_retirement_probabilities(prob_middle_players_retire, active_middle_players, min_age=18, max_age=40, legend_name = "Middle Players")
plot_retirement_probabilities(prob_top_players_retire, active_top_players, min_age=18, max_age=40, legend_name="Top Players")

Let's see these 3 curves together on the same graph, with the curve containing all players. The aim is to see whether it is necessary to take into account the three groups or not. 

In [ ]:
age_min_limit = 16
age_max_limit = 40

groups = {
    f"Top ({(1-top_quantile)*100:.0f}%)": (prob_top_players_retire, "#e59718c5"),
    f"Middle ({(top_quantile-least_quantile)*100:.0f}%)": (prob_middle_players_retire, "#60a215e8"),
    f"Bottom ({least_quantile*100:.0f}%)": (prob_bottom_players_retire, "#1484cfe8")
    }

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

# plot of retirement probability for each group 
for group_legend, (prob_data, color) in groups.items():
        prob_players_retire = prob_data.loc[age_min_limit:age_max_limit]

        sns.scatterplot(x=prob_players_retire.index, y=prob_players_retire.values,
                        color=color, edgecolor="black", linewidth=1.5, s=75,
                       zorder=2, label=group_legend)
        
        plt.plot(prob_players_retire.index, prob_players_retire.values, color=color,
                 linewidth=1.5, linestyle="--", zorder=1)

# plot of the retirement probability for all players together
prob_all_players_retire_filtered = prob_all_players_retire.loc[age_min_limit:age_max_limit]


sns.scatterplot(x=prob_all_players_retire_filtered.index, y=prob_all_players_retire_filtered.values, 
                color="#bdc3c7", edgecolor=None, 
                linewidth=1.5, s=75, alpha=0.8, 
                zorder=0, label=f"All Players")
       
plt.plot(prob_all_players_retire_filtered.index, prob_all_players_retire_filtered.values, color="#bdc3c7",
                 linewidth=1.5, linestyle="-", zorder=0, alpha=0.15)


plt.title("Observed Probability of Retirement by Age and Category", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age",fontsize=18)
plt.ylabel("Probability to retire",fontsize=18)
plt.legend(frameon=True, fontsize=13, title="Category", loc="upper center")
plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# get the parameters of the fit for the BOTTOM players

min_age_fit_bottom = 16
max_age_fit_bottom = 38

fits_results_bottom = fit_retirement_data(prob_bottom_players_retire, min_age_fit=min_age_fit_bottom, max_age_fit=max_age_fit_bottom)

popt_lin_bottom = fits_results_bottom["linear"]["params"]
popt_quad_bottom = fits_results_bottom["quadratic"]["params"]
popt_exp_bottom = fits_results_bottom["exp"]["params"]

plot_fit_comparisons(prob_bottom_players_retire, min_age_fit_bottom, max_age_fit_bottom, popt_lin=popt_lin_bottom, popt_quad=popt_quad_bottom, popt_exp=popt_exp_bottom, legend_name="Bottom Players")

In [ ]:
# get the parameters of the fit for the MIDDLE players
min_age_fit_middle = 16
max_age_fit_middle = 31

fits_results_middle = fit_retirement_data(prob_middle_players_retire, min_age_fit=min_age_fit_middle, max_age_fit=max_age_fit_middle)

popt_lin_middle = fits_results_middle["linear"]["params"]
popt_quad_middle = fits_results_middle["quadratic"]["params"]
popt_exp_middle = fits_results_middle["exp"]["params"]

plot_fit_comparisons(prob_middle_players_retire, min_age_fit_middle, max_age_fit_middle, popt_lin=popt_lin_middle, popt_quad=popt_quad_middle, popt_exp=popt_exp_middle, legend_name="Middle Players")

In [ ]:
# get the parameters of the fit for the TOP players
min_age_fit_top = 15
max_age_fit_top = 39

fits_results_top = fit_retirement_data(prob_top_players_retire, min_age_fit=min_age_fit_top, max_age_fit=max_age_fit_top)

popt_lin_top = fits_results_top["linear"]["params"]
popt_quad_top = fits_results_top["quadratic"]["params"]
popt_exp_top = fits_results_top["exp"]["params"]

plot_fit_comparisons(prob_top_players_retire, min_age_fit_top, max_age_fit_top, popt_lin=popt_lin_top, popt_quad=popt_quad_top, popt_exp=popt_exp_top, legend_name="Top Players")

In [ ]:
plt.figure(figsize=(16, 9))
sns.set_style("ticks") 
plt.rcParams.update({'font.family': 'serif'})

# plot of retirement probability for each group 
for group_legend, (prob_data, color) in groups.items():
        prob_players_retire = prob_data.loc[age_min_limit:age_max_limit]

        sns.scatterplot(x=prob_players_retire.index, y=prob_players_retire.values,
                        color=color, edgecolor="black", linewidth=1.5, s=75,
                       zorder=2, label=group_legend)
        
        plt.plot(prob_players_retire.index, prob_players_retire.values, color=color,
                 linewidth=1.5, linestyle="--", zorder=1)

# plot the model fits on it 
ages_ext = np.linspace(15, 45, 100) 
plt.plot(ages_ext, linear_model(ages_ext, *popt_lin_bottom), color="#1484cfe8", linewidth=3, zorder=4)
plt.plot(ages_ext, linear_model(ages_ext, *popt_lin_middle), color="#60a215e8", linewidth=3, zorder=4)
plt.plot(ages_ext, exp_model(ages_ext, *popt_exp_top), color="#e59718c5", linewidth=3, zorder=4) 


plt.title("Observed Probability of Retirement by Age and Category", fontsize=25, weight="bold", pad=35)

plt.xlabel("Player Age",fontsize=18)
plt.ylabel("Probability to retire",fontsize=18)
plt.legend(frameon=True, fontsize=13, title="Category", loc="upper center")
plt.grid(visible=True, which="major", axis="both", linewidth=0.5, alpha=0.5)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
from scipy.optimize import curve_fit

# polynom of degree 2 (with 3 parameters)
def polynom_2(x,a,b,c):
    return a*x**2+b*x+c

# polynom of degree 3 (with 4 parameters)
def polynom_3(x,a,b,c,d):
    return a*x**3+b*x**2+c*x+d

ages = aging_median["age"]
median_diff = aging_median["log10_strength_diff"]

popt_2, pcov_2 = curve_fit(polynom_2, ages, median_diff)
perr_2 = np.sqrt(np.diag(pcov_2))

popt_3, pcov_3 = curve_fit(polynom_3, ages, median_diff)
perr_3 = np.sqrt(np.diag(pcov_3))

print("-- Polynom of degree 2: ax^2 + bx + c ---")
print(f"Parameters: a= {popt_2[0]:.2e} ± {perr_2[0]:.2e} , b= {popt_2[1]:.4f} ± {perr_2[1]:.2e}, c= {popt_2[2]:.4f} ± {perr_2[2]:.2e}")

print("\n-- Polynom of degree 3: ax^3 + bx^2 + cx + d ---")
print(f"Parameters: a= {popt_3[0]:.2e} ± {perr_3[0]:.2e}, b= {popt_3[1]:.2e} ± {perr_3[1]:.2e}, c= {popt_3[2]:.2f} ± {perr_3[2]:.2e}, d= {popt_3[3]:.2f} ± {perr_3[3]:.2e}")

#### Conclusion on the Player Retirement Process

* Global Model with All Players included: the empirical data from age 16 to 39 shows an almost-linear increase. For simplicity, we will use the linear model if needed.
* Stratified Models: the bottom players directly face high probability, and the linear fit captures well this high probability and keeps it simple. The middle players also follow a linear trend. For the elite players, the most accurate choice is the exponential mode, with near-zero retirement probability in the 20s. However, as they reach their 30s, the retirement risk accelerates.



In [ ]:
# saving all the parameters in the .json file

stratified_models = {
    "quantiles": [least_quantile, top_quantile],
    "categories_limit": [lower_limit, upper_limit],
    
    "bottom": {
        "calibration_ages": [min_age_fit_bottom, max_age_fit_bottom],
        "linear": {
            "params": popt_lin_bottom.tolist(),
            "equation": "p[0]*(age-18) + p[1]"
            },
        
        "quadratic": {
            "params": popt_quad_bottom.tolist(),
            "equation": "p[0]*(age-18)^2 + p[1]*(age-18) + p[2]"
            },
        
        "exp": {
            "params": popt_exp_bottom.tolist() if popt_exp_bottom is not None else None,
            "equation": "p[0]*exp(p[1]*(age-18)) + p[2]"
            }
    },

    "middle": {
        "calibration_ages": [min_age_fit_middle, max_age_fit_middle],
        "linear": {
            "params": popt_lin_middle.tolist(),
            "equation": "p[0]*(age-18) + p[1]"
            },
        
        "quadratic": {
            "params": popt_quad_middle.tolist(),
            "equation": "p[0]*(age-18)^2 + p[1]*(age-18) + p[2]"
            },
        
        "exp": {
            "params": popt_exp_middle.tolist() if popt_exp_middle is not None else None,
            "equation": "p[0]*exp(p[1]*(age-18)) + p[2]"
            }
    },
    "top": {
        "calibration_ages": [min_age_fit_top, max_age_fit_top],
        "linear": {
            "params": popt_lin_top.tolist(),
            "equation": "p[0]*(age-18) + p[1]"
            },
        
        "quadratic": {
            "params": popt_quad_top.tolist(),
            "equation": "p[0]*(age-18)^2 + p[1]*(age-18) + p[2]"
            },
        
        "exp": {
            "params": popt_exp_top.tolist() if popt_exp_top is not None else None,
            "equation": "p[0]*exp(p[1]*(age-18)) + p[2]"
            }
    }
}

# saving the parameters in the .json file created earlier
config_path = "../config/simulation_params.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f: # opening the file in read mode
        config_data = json.load(f)
else:
    config_data = {}

if "retirement_params" not in config_data:
    config_data["retirement_params"] = {}

config_data["retirement_params"]["stratified_models"] = stratified_models

os.makedirs(os.path.dirname(config_path), exist_ok=True)


with open(config_path, "w") as f: # opening the file in writing mode
    json.dump(config_data, f, indent=3)

## Sources

$\textbf{1.2. Stationarity of the Number of New Players (Quantity)}$

- Ghasemi A., & Zahediasl S. (2012). $\textit{Normality Tests for Statistical Analysis: A Guide for Non-Statisticians}$, International Journal of Endocrinology and Metabolism, 10(2), 486–489.
https://pmc.ncbi.nlm.nih.gov/articles/PMC3693611/ $\newline$
(for justifying the combination of graphic inspection + statistical tests for normality, because if statistical tests with a p-value only can lead to erroneous conclusions)
- Razali, N. M., & Wah, Y. B. (2011). $\textit{Power comparisons of Shapiro-Wilk, Kolmogorov-Smirnov, Lilliefors and Anderson-Darling tests.}$ Journal of Statistical Modeling and Analytics, 2(1), 21–33. https://www.nrc.gov/docs/ml1714/ml17143a100.pdf $\newline$ (for justifying the choice of Shapiro-Wilk test for normality, small samples)
- Shapiro, S. S., & Wilk, M. B. (1965). $\textit{An Analysis of Variance Test for Normality (Complete Samples).}$ Biometrika, 52(3/4), 591–611. https://doi.org/10.2307/2333709 $\newline$
(for the original paper introducing the Shapiro-Wilk test)
- Wilk, M. B., & Gnanadesikan, R. (1968). Probability Plotting Methods for the Analysis of Data. Biometrika, 55(1), 1–17. https://doi.org/10.2307/2334448 $\newline$
(for the original paper introducing the Q-Q plot)


